

# AURORA: Advanced Unified Retrieval-Optimized Reasoning Architecture

## A Principal-Level Technical Report on State-of-the-Art Retrieval-Augmented Generation Pipeline Engineering

---

## 1. Document Scope and Architectural Premise

This report formalizes the complete execution stack of a production-grade Retrieval-Augmented Generation (RAG) system, articulated as a typed, multi-stage pipeline with mathematically grounded optimization at every boundary. Each stage—indexing, pre-retrieval, retrieval, and post-retrieval—is specified with state-of-the-art algorithms, formal loss functions, provenance-tracked data flow, and measurable quality gates. The objective is to transform the naive retrieve-and-read paradigm into a deterministic, verifiable, and continuously improvable agentic retrieval architecture.

The canonical RAG pipeline is formalized as a composition of operators:

$$
\mathcal{R}_{\text{RAG}}(q) = \mathcal{G}\!\Big(\mathcal{P}\!\big(q,\; \mathcal{F}_{\text{post}}\!\left(\mathcal{S}\!\left(\mathcal{F}_{\text{pre}}(q),\; \mathcal{I}(\mathcal{D})\right)\right)\big)\Big)
$$

where $q$ is the user query, $\mathcal{D}$ is the document corpus, $\mathcal{I}$ is the indexing operator, $\mathcal{F}_{\text{pre}}$ is the pre-retrieval transformation, $\mathcal{S}$ is the retrieval/search operator, $\mathcal{F}_{\text{post}}$ is the post-retrieval optimization operator, $\mathcal{P}$ is the prompt compiler, and $\mathcal{G}$ is the generative LLM.

**Pipeline Stage Map:**

| Stage | Operators | Optimization Target |
|---|---|---|
| **Indexing** | Data extraction, cleaning, transformation, chunking | Retrieval precision via structured, semantically coherent index units |
| **Pre-Retrieval** | Query rewriting, decomposition, routing | Query-index alignment, intent disambiguation, source selection |
| **Retrieval** | Metadata filtering, outlier exclusion, hybrid search, embedding fine-tuning | Recall, precision, latency, relevance under token budget |
| **Post-Retrieval** | Re-ranking, context compression/enhancement, prompt engineering, LLM fine-tuning | Generation accuracy, faithfulness, hallucination suppression |

---

## 2. Indexing Optimization: Deterministic Corpus Preparation

### 2.1 Data Pre-Processing Pipeline

The indexing stage transforms a heterogeneous corpus $\mathcal{D} = \{d_1, d_2, \dots, d_N\}$ into a set of retrieval-ready, provenance-tagged index units. This is not ad hoc ETL; it is a typed, reproducible data pipeline with schema enforcement at every stage.

**Formal Pipeline Definition:**

$$
\mathcal{I}(\mathcal{D}) = \texttt{Chunk} \circ \texttt{Transform} \circ \texttt{Clean} \circ \texttt{Parse} \circ \texttt{Extract}(\mathcal{D})
$$

Each operator is a typed function with explicit input/output schemas, provenance metadata propagation, and idempotent execution guarantees.

#### 2.1.1 Extraction and Parsing

For each document $d_i \in \mathcal{D}$ with format type $\tau(d_i) \in \{\texttt{PDF}, \texttt{HTML}, \texttt{DOCX}, \texttt{MD}, \texttt{XLSX}, \texttt{IMG}, \dots\}$, the extraction operator selects a format-specific parser:

$$
\texttt{Parse}(d_i) = \phi_{\tau(d_i)}(d_i) \rightarrow (T_i, M_i)
$$

where $T_i$ is the extracted text/structure and $M_i$ is the extracted metadata (author, timestamp, source URI, format, section hierarchy).

**State-of-the-art parsing strategies:**

For image-embedded documents (scanned PDFs, infographics), the SOTA approach replaces legacy OCR with vision-language embedding models. Given a document page rendered as image $I_k$, a model such as ColPali computes a multi-vector embedding directly:

$$
\mathbf{E}_{\text{page}} = \text{ColPali}(I_k) = \{\mathbf{v}_1, \mathbf{v}_2, \dots, \mathbf{v}_P\}, \quad \mathbf{v}_j \in \mathbb{R}^d
$$

where each $\mathbf{v}_j$ corresponds to a patch-level representation. Retrieval scoring against a query $q$ uses late interaction:

$$
\text{score}(q, I_k) = \sum_{i=1}^{|q|} \max_{j \in \{1, \dots, P\}} \; \mathbf{q}_i^\top \mathbf{v}_j
$$

This eliminates OCR error propagation entirely and preserves layout-sensitive information (tables, charts, spatial relationships) that text-only pipelines destroy.

#### 2.1.2 Cleaning and Noise Reduction

The cleaning operator $\texttt{Clean}(T_i, M_i) \rightarrow (\hat{T}_i, M_i)$ applies deterministic transformations:

- Boilerplate removal via structural classifier (headers, footers, navigation elements scored by positional entropy)
- Unicode normalization (NFC form), whitespace canonicalization
- Deduplication via MinHash locality-sensitive hashing at paragraph level:

$$
\text{Jaccard}(A, B) \approx \frac{|\text{MinHash}(A) \cap \text{MinHash}(B)|}{|\text{MinHash}(A) \cup \text{MinHash}(B)|} > \theta_{\text{dup}} \implies \text{merge}
$$

where $\theta_{\text{dup}}$ is a configurable deduplication threshold (typical range $[0.85, 0.95]$).

- Missing value imputation for structured fields (interpolation for numeric, null-token for categorical)
- Structural integrity validation: schema conformance checks on table cell relationships, list nesting depth, cross-reference resolution

#### 2.1.3 Transformation and Standardization

All cleaned outputs are projected into a canonical document element schema:

$$
\texttt{Transform}(\hat{T}_i, M_i) \rightarrow \{e_1^{(i)}, e_2^{(i)}, \dots, e_{K_i}^{(i)}\}
$$

where each element $e_k^{(i)}$ is a typed record:

```
DocumentElement {
  element_id:    UUID
  doc_id:        UUID
  element_type:  enum {TITLE, SECTION, PARAGRAPH, TABLE, LIST, CODE, IMAGE_REF}
  content:       string
  metadata:      MetadataRecord  // author, timestamp, source_uri, section_path, ...
  lineage:       ProvenanceChain // extraction_method, parser_version, cleaning_ops
  position:      (page, offset, depth)
}
```

This typed schema ensures downstream operators (chunking, embedding, retrieval) operate on structurally homogeneous units with full provenance traceability.

---

### 2.2 Chunking Strategies: Formal Analysis and SOTA Algorithms

Chunking partitions a sequence of document elements into retrieval units $\{c_1, c_2, \dots, c_M\}$. The fundamental trade-off is between **retrieval precision** (smaller chunks match queries more exactly) and **contextual completeness** (larger chunks preserve reasoning context for generation).

#### 2.2.1 Formal Chunking Objective

Define chunk quality as a function of retrieval relevance and generative utility:

$$
\mathcal{Q}(c) = \underbrace{\text{Rel}(c, q)}_{\text{retrieval precision}} \cdot \underbrace{\text{Suf}(c, a)}_{\text{answer sufficiency}} \cdot \underbrace{\text{Coh}(c)}_{\text{internal coherence}}
$$

The chunking optimization problem over a document $D = (t_1, t_2, \dots, t_L)$ (token sequence of length $L$) seeks a partition $\Pi^* = \{c_1, \dots, c_M\}$ that maximizes expected retrieval quality under a size constraint:

$$
\Pi^* = \arg\max_{\Pi} \; \mathbb{E}_{q \sim \mathcal{Q}_{\text{dist}}} \left[ \sum_{c \in \Pi} \mathcal{Q}(c) \right] \quad \text{s.t.} \quad \forall c \in \Pi: \; |c| \leq \ell_{\max}
$$

where $|c|$ is the token count of chunk $c$ and $\ell_{\max}$ is the maximum chunk size.

#### 2.2.2 Chunking Strategy Taxonomy and SOTA Selection

**Strategy 1: Fixed-Size Chunking with Overlap**

Partition into windows of size $w$ with overlap $o$:

$$
c_k = (t_{k(w-o)+1}, \;\dots,\; t_{k(w-o)+w}), \quad k = 0, 1, \dots
$$

Token budget per chunk: $w$. Overlap ratio: $\rho = o / w$.

*Limitations:* Ignores semantic boundaries; splits entities, sentences, and logical units arbitrarily. Used only as a baseline or fallback.

**Strategy 2: Recursive Hierarchical Chunking**

Apply a priority-ordered separator hierarchy $\mathcal{S} = (s_1, s_2, \dots, s_n)$ (e.g., $s_1 = \texttt{section\_break}$, $s_2 = \texttt{paragraph\_break}$, $s_3 = \texttt{sentence\_break}$). Split at the highest-priority separator first; if any resulting chunk exceeds $\ell_{\max}$, recursively apply the next separator.

```
ALGORITHM 1: RecursiveHierarchicalChunk
─────────────────────────────────────────────────────────────
Input:  text T, separators S = [s₁, s₂, ..., sₙ], max_size ℓ_max
Output: chunks C

1   function RECHUNK(T, S, depth):
2       if |T| ≤ ℓ_max or depth ≥ |S|:
3           return [T]
4       segments ← SPLIT(T, S[depth])
5       C ← []
6       buffer ← ""
7       for seg in segments:
8           if |buffer + seg| ≤ ℓ_max:
9               buffer ← buffer + seg
10          else:
11              if buffer ≠ "":
12                  C ← C ∪ [buffer]
13              if |seg| > ℓ_max:
14                  C ← C ∪ RECHUNK(seg, S, depth + 1)
15                  buffer ← ""
16              else:
17                  buffer ← seg
18      if buffer ≠ "":
19          C ← C ∪ [buffer]
20      return C
21
22  return RECHUNK(T, S, 0)
```

**Strategy 3: Semantic Chunking via Embedding Discontinuity Detection**

This SOTA approach detects semantic boundaries by measuring embedding similarity between consecutive textual units $u_i$.

1. Segment text into atomic units (sentences): $U = (u_1, u_2, \dots, u_n)$.
2. Embed each unit: $\mathbf{e}_i = \text{Embed}(u_i) \in \mathbb{R}^d$.
3. Compute pairwise cosine similarity between consecutive embeddings:

$$
\text{sim}(i) = \frac{\mathbf{e}_i \cdot \mathbf{e}_{i+1}}{\|\mathbf{e}_i\| \|\mathbf{e}_{i+1}\|}
$$

4. Detect breakpoints where similarity drops below a threshold derived from the distribution of similarities:

$$
\text{break}(i) = \mathbb{1}\!\left[\text{sim}(i) < \mu_{\text{sim}} - \beta \cdot \sigma_{\text{sim}}\right]
$$

where $\mu_{\text{sim}}$ and $\sigma_{\text{sim}}$ are the mean and standard deviation of the similarity sequence, and $\beta$ is a sensitivity hyperparameter (typically $\beta \in [1.0, 2.0]$).

```
ALGORITHM 2: SemanticChunking
─────────────────────────────────────────────────────────────
Input:  sentences U = [u₁, ..., uₙ], embed_model, β, ℓ_max
Output: chunks C

1   E ← [embed_model(uᵢ) for uᵢ in U]
2   sim ← [cosine(E[i], E[i+1]) for i in 1..n-1]
3   μ ← mean(sim);  σ ← std(sim)
4   threshold ← μ - β · σ
5   breakpoints ← {i : sim[i] < threshold}
6   C ← [];  buffer ← []
7   for i in 1..n:
8       buffer.append(uᵢ)
9       if i ∈ breakpoints or |tokens(buffer)| ≥ ℓ_max:
10          C.append(join(buffer))
11          buffer ← []
12  if buffer ≠ []:
13      C.append(join(buffer))
14  return C
```

**Strategy 4: Late Chunking (SOTA — Context-Preserving)**

Late chunking reverses the conventional embed-after-chunk order. It processes the entire document through a long-context embedding model first, obtaining token-level contextualized representations, and then applies boundary segmentation to produce chunk embeddings that encode document-wide context.

Given document token sequence $(t_1, \dots, t_L)$ and a long-context encoder $\mathcal{E}_{\text{long}}$:

1. Compute contextualized token embeddings:

$$
(\mathbf{h}_1, \mathbf{h}_2, \dots, \mathbf{h}_L) = \mathcal{E}_{\text{long}}(t_1, t_2, \dots, t_L)
$$

2. Determine chunk boundaries $B = \{b_0, b_1, \dots, b_M\}$ using any boundary detection method (structural, semantic, or fixed).

3. Compute chunk embedding by mean-pooling over the contextualized token representations within each boundary:

$$
\mathbf{c}_k = \frac{1}{b_k - b_{k-1}} \sum_{j=b_{k-1}+1}^{b_k} \mathbf{h}_j
$$

The critical advantage is that each $\mathbf{h}_j$ is conditioned on the full document via the transformer's self-attention, so $\mathbf{c}_k$ retains cross-chunk contextual information that early-chunking approaches destroy.

```
ALGORITHM 3: LateChunking
─────────────────────────────────────────────────────────────
Input:  document D = [t₁, ..., t_L], long_ctx_encoder, boundary_fn
Output: chunk_embeddings CE, chunk_texts CT

1   H ← long_ctx_encoder(D)          // H ∈ ℝ^{L × d}, contextualized
2   boundaries ← boundary_fn(D)      // [b₀=0, b₁, ..., b_M=L]
3   CE ← [];  CT ← []
4   for k in 1..M:
5       cₖ ← mean(H[b_{k-1}+1 : bₖ])    // mean-pool contextualized repr
6       CE.append(cₖ)
7       CT.append(D[b_{k-1}+1 : bₖ])
8   return CE, CT
```

**Strategy 5: Agentic Chunking (LLM-Driven Proposition Extraction)**

An LLM decomposes text into self-contained atomic propositions (facts, claims, definitions), each independently meaningful and retrievable. This is the highest-fidelity but most computationally expensive strategy.

$$
\texttt{AgenticChunk}(D) = \text{LLM}_{\text{decompose}}\!\left(D,\; \texttt{prompt}_{\text{propositionize}}\right) \rightarrow \{p_1, p_2, \dots, p_K\}
$$

Each $p_i$ is a decontextualized proposition: all pronouns resolved, all implicit references made explicit, all temporal/spatial anchors attached.

**Chunking Strategy Selection Matrix:**

| Document Class | Recommended Strategy | Rationale |
|---|---|---|
| Structured (HTML, MD, code) | Document-based + recursive | Natural section boundaries provide semantic coherence |
| Long-form prose (articles, reports) | Semantic chunking | Embedding discontinuity aligns with topic shifts |
| Dense technical (patents, legal) | Agentic (proposition extraction) | High entity density requires decontextualization |
| Multi-modal (scanned docs) | Late chunking with ColPali | Preserves layout-level context across page regions |
| High-volume, uniform (logs, records) | Fixed-size with overlap | Cost-efficient; content is structurally homogeneous |

---

## 3. Pre-Retrieval Optimization: Query Engineering

Pre-retrieval optimization transforms the raw user query $q$ into one or more optimized search queries $\hat{q}$ that maximize retrieval performance against the index.

### 3.1 Query Rewriting (Rewrite-Retrieve-Read)

The Rewrite-Retrieve-Read paradigm interposes a learned rewriting function between the user and the retriever:

$$
\hat{q} = \mathcal{R}_{\text{rewrite}}(q, \mathcal{H}) = \text{LLM}\!\left(q,\; \texttt{prompt}_{\text{rewrite}},\; \mathcal{H}\right)
$$

where $\mathcal{H}$ is optional conversational history providing disambiguation context.

The rewriter is optimized to maximize downstream retrieval recall. In the SOTA formulation (RRR — Ma et al., 2023), the rewriter is trained with reinforcement learning where the reward signal is the quality of the final generated answer:

$$
\mathcal{L}_{\text{rewrite}} = -\mathbb{E}_{q \sim \mathcal{Q}} \left[ R\!\left(\mathcal{G}\!\left(\mathcal{P}\!\left(\hat{q},\; \mathcal{S}(\hat{q})\right)\right),\; a^*\right) \right]
$$

where $R(\cdot, a^*)$ is a reward function measuring answer quality against ground truth $a^*$.

```
ALGORITHM 4: RewriteRetrieveRead
─────────────────────────────────────────────────────────────
Input:  query q, history H, rewriter_llm, retriever, generator_llm
Output: response r

1   q̂ ← rewriter_llm(REWRITE_PROMPT, q, H)
2   docs ← retriever.search(q̂, top_k=K_initial)
3   context ← format_context(docs)
4   r ← generator_llm(GEN_PROMPT, q, context)
5   return r
```

### 3.2 Query Expansion (Multi-Query Generation)

Query expansion generates $n$ semantically diverse reformulations to increase recall coverage across the embedding space:

$$
\{q_1', q_2', \dots, q_n'\} = \text{LLM}_{\text{expand}}(q, \texttt{prompt}_{\text{diversify}})
$$

Retrieved document sets are aggregated via union:

$$
\mathcal{C}_{\text{expanded}} = \bigcup_{i=1}^{n} \mathcal{S}(q_i', K)
$$

Because the union inflates result count, a downstream re-ranking stage is mandatory (Section 5.1). Deduplication is applied by document ID before re-ranking.

The SOTA variant, **RAG-Fusion** (Raudaschl, 2023), combines multi-query expansion with **Reciprocal Rank Fusion (RRF)** scoring:

$$
\text{RRF}(d) = \sum_{i=1}^{n} \frac{1}{\kappa + \text{rank}_i(d)}
$$

where $\text{rank}_i(d)$ is the rank of document $d$ in the result list for query $q_i'$, and $\kappa$ is a smoothing constant (typically $\kappa = 60$).

```
ALGORITHM 5: RAGFusionExpansion
─────────────────────────────────────────────────────────────
Input:  query q, expansion_llm, retriever, n_expansions, κ=60
Output: fused_ranked_docs

1   queries ← [q] ∪ expansion_llm(EXPAND_PROMPT, q, n=n_expansions)
2   ranked_lists ← []
3   for q' in queries:
4       results ← retriever.search(q', top_k=K_overretrieve)
5       ranked_lists.append(results)
6   doc_scores ← defaultdict(float)
7   for i, rlist in enumerate(ranked_lists):
8       for rank, doc in enumerate(rlist, start=1):
9           doc_scores[doc.id] += 1.0 / (κ + rank)
10  fused_ranked_docs ← sort(doc_scores, descending=True)
11  return fused_ranked_docs
```

### 3.3 Query Decomposition

For complex, multi-faceted queries, decomposition breaks the original query into independently retrievable sub-queries, each targeting a specific information need.

$$
\{q_{\text{sub}}^{(1)}, q_{\text{sub}}^{(2)}, \dots, q_{\text{sub}}^{(m)}\} = \text{LLM}_{\text{decompose}}(q, \texttt{prompt}_{\text{decompose}})
$$

Sub-query results are retrieved independently (parallelizable), then synthesized:

$$
r = \mathcal{G}\!\left(q,\; \bigoplus_{j=1}^{m} \left(q_{\text{sub}}^{(j)},\; \mathcal{S}(q_{\text{sub}}^{(j)})\right)\right)
$$

where $\bigoplus$ denotes structured aggregation (concatenation with sub-query labels).

The SOTA extension enriches decomposition with **keyword extraction** and **metadata filter extraction** per sub-query:

$$
\text{LLM}_{\text{decompose}}(q) \rightarrow \left\{\left(q_{\text{sub}}^{(j)},\; \mathcal{K}^{(j)},\; \mathcal{F}^{(j)}\right)\right\}_{j=1}^{m}
$$

where $\mathcal{K}^{(j)}$ is a keyword set for sparse retrieval augmentation and $\mathcal{F}^{(j)}$ is a metadata filter predicate (e.g., `{date > 2024-01, domain = "cardiology"}`).

```
ALGORITHM 6: StructuredQueryDecomposition
─────────────────────────────────────────────────────────────
Input:  complex_query q, decompose_llm, retriever, generator_llm
Output: final_response r

1   decomposition ← decompose_llm(DECOMPOSE_PROMPT, q)
2   // decomposition: [{subquery, keywords, metadata_filters}, ...]
3   sub_results ← []
4   PARALLEL for each (sq, kw, mf) in decomposition:
5       dense_hits ← retriever.semantic_search(sq, top_k=K)
6       sparse_hits ← retriever.keyword_search(kw, top_k=K)
7       filtered ← apply_metadata_filters(dense_hits ∪ sparse_hits, mf)
8       sub_results.append({subquery: sq, evidence: filtered})
9   aggregated_context ← format_subquery_evidence(sub_results)
10  r ← generator_llm(SYNTHESIS_PROMPT, q, aggregated_context)
11  return r
```

### 3.4 Query Routing

Query routing selects the optimal retrieval pipeline, index, or tool based on query characteristics. This is the entry point for agentic RAG architectures.

Define a routing function:

$$
\pi(q) = \arg\max_{r \in \mathcal{R}} \; P(r \mid q, \theta_{\text{router}})
$$

where $\mathcal{R}$ is the set of available retrieval routes (vector indices, keyword indices, SQL databases, calculators, web search APIs, specialized agents) and $\theta_{\text{router}}$ parameterizes the router (an LLM classifier, a fine-tuned small model, or a rule-based system).

**SOTA routing architecture** implements a **single-agent RAG router** with tool-use capability:

```
ALGORITHM 7: AgenticQueryRouter
─────────────────────────────────────────────────────────────
Input:  query q, route_registry R, router_agent
Output: response r

1   // Route registry R: [{route_id, description, schema, handler}, ...]
2   route_decision ← router_agent.decide(q, R.descriptions())
3   // route_decision: {selected_routes: [...], query_transforms: [...]}
4   results ← []
5   for (route, transformed_q) in zip(route_decision.selected_routes,
6                                      route_decision.query_transforms):
7       handler ← R.get_handler(route)
8       result ← handler.execute(transformed_q, timeout=DEADLINE)
9       results.append({route: route, evidence: result})
10  context ← merge_route_results(results)
11  r ← generator_llm(GEN_PROMPT, q, context)
12  return r
```

The router may select multiple routes for a single query (fan-out), apply different query transformations per route, and merge results with provenance tags indicating source route.

---

## 4. Retrieval Optimization: Precision Engineering

### 4.1 Metadata Filtering

Metadata filtering applies predicate-based constraints before or during vector search to reduce the candidate set:

$$
\mathcal{C}_{\text{filtered}} = \{d \in \mathcal{C} : \mathcal{F}(d.M) = \texttt{true}\}
$$

where $\mathcal{F}$ is a Boolean predicate over metadata fields $d.M$.

In production vector databases, metadata filtering is implemented as **pre-filtering** (filter then search) or **post-filtering** (search then filter). Pre-filtering is preferred for selectivity ratio $\sigma$:

$$
\sigma = \frac{|\mathcal{C}_{\text{filtered}}|}{|\mathcal{C}|}, \quad \text{pre-filter if } \sigma > \sigma_{\min} \text{ (typically } 0.01\text{)}
$$

If $\sigma$ is extremely low (very selective filter), the filtered set may be too small for meaningful vector search. In this case, exact retrieval over the filtered set is more efficient.

**Time-aware metadata scoring** augments relevance with freshness:

$$
\text{score}_{\text{final}}(d) = \alpha \cdot \text{sim}(q, d) + (1 - \alpha) \cdot \text{freshness}(d)
$$

where freshness is defined via exponential decay:

$$
\text{freshness}(d) = \exp\!\left(-\lambda \cdot (t_{\text{now}} - t_d)\right)
$$

with $\lambda$ controlling the decay rate and $t_d$ being the document timestamp.

### 4.2 Vector Search Outlier Exclusion

Three SOTA techniques for managing retrieval quality beyond naive top-$k$:

**4.2.1 Distance Thresholding**

Apply a hard cutoff on the distance metric:

$$
\mathcal{C}_{\text{thresh}} = \{d \in \text{top}_K(q) : \text{dist}(q, d) \leq \delta_{\max}\}
$$

The threshold $\delta_{\max}$ must be calibrated per embedding model and distance metric (cosine, L2, dot product). Calibration uses a validation set of known-relevant and known-irrelevant pairs to determine the optimal operating point on the precision-recall curve.

**4.2.2 Autocut (Adaptive Cluster-Based Truncation)**

Autocut analyzes the distribution of distances in the result set and identifies natural breakpoints:

$$
\Delta_i = \text{dist}(q, d_{i+1}) - \text{dist}(q, d_i), \quad i = 1, \dots, K-1
$$

A breakpoint is detected at position $i^*$ where:

$$
i^* = \arg\max_i \; \Delta_i \quad \text{s.t.} \quad \Delta_{i^*} > \mu_\Delta + \gamma \cdot \sigma_\Delta
$$

All results beyond $i^*$ are discarded. The parameter $\gamma$ controls sensitivity (typically $\gamma \in [1.0, 2.5]$).

```
ALGORITHM 8: AutocutOutlierExclusion
─────────────────────────────────────────────────────────────
Input:  ranked_results R = [(d₁, dist₁), ..., (d_K, dist_K)], γ
Output: filtered_results

1   deltas ← [dist_{i+1} - distᵢ for i in 1..K-1]
2   μ_Δ ← mean(deltas);  σ_Δ ← std(deltas)
3   cutoff_threshold ← μ_Δ + γ · σ_Δ
4   cut_index ← K    // default: no cut
5   for i in 1..|deltas|:
6       if deltas[i] > cutoff_threshold:
7           cut_index ← i + 1
8           break
9   return R[1 : cut_index]
```

**4.2.3 Percentile-Based Dynamic Thresholding**

For distributions with heavy tails, use the $p$-th percentile of the distance distribution from a calibration corpus as the threshold:

$$
\delta_{\max} = \text{Percentile}_p\!\left(\{\text{dist}(q_j, d_j^+)\}_{j=1}^{N_{\text{cal}}}\right)
$$

where $d_j^+$ is the ground-truth relevant document for calibration query $q_j$, and $p$ is typically 95 or 99.

### 4.3 Hybrid Search

Hybrid search fuses dense (semantic) and sparse (lexical) retrieval to capture both conceptual similarity and exact term matching.

**Dense retrieval** score (e.g., cosine similarity):

$$
s_{\text{dense}}(q, d) = \frac{\mathbf{e}_q \cdot \mathbf{e}_d}{\|\mathbf{e}_q\| \|\mathbf{e}_d\|}
$$

**Sparse retrieval** score (e.g., BM25):

$$
s_{\text{BM25}}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t, d) \cdot (k_1 + 1)}{f(t, d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}
$$

where $f(t, d)$ is the term frequency of term $t$ in document $d$, $|d|$ is the document length, $\text{avgdl}$ is the average document length, and $k_1$, $b$ are BM25 hyperparameters.

**Fusion via the $\alpha$ parameter:**

$$
s_{\text{hybrid}}(q, d) = \alpha \cdot \hat{s}_{\text{dense}}(q, d) + (1 - \alpha) \cdot \hat{s}_{\text{BM25}}(q, d)
$$

where $\hat{s}$ denotes min-max normalized scores (both mapped to $[0, 1]$), and $\alpha \in [0, 1]$ controls the weighting:

| $\alpha$ | Behavior |
|---|---|
| $1.0$ | Pure semantic search |
| $0.0$ | Pure keyword (BM25) search |
| $0.5$ | Equal weighting |
| $0.7$ | Semantic-dominant hybrid (common SOTA default) |

**SOTA fusion: Reciprocal Rank Fusion (RRF)** provides a rank-based alternative that avoids score normalization issues:

$$
\text{RRF}(d) = \sum_{l \in \{\text{dense}, \text{sparse}\}} \frac{w_l}{\kappa + \text{rank}_l(d)}
$$

where $w_l$ are per-retriever weights and $\kappa$ is the smoothing constant.

```
ALGORITHM 9: HybridRetrievalWithRRF
─────────────────────────────────────────────────────────────
Input:  query q, dense_retriever, sparse_retriever, α, κ=60, K
Output: hybrid_results

1   dense_results ← dense_retriever.search(q, top_k=K_over)
2   sparse_results ← sparse_retriever.search(q, top_k=K_over)
3   // Option A: Score-level fusion
4   norm_dense ← min_max_normalize(dense_results.scores)
5   norm_sparse ← min_max_normalize(sparse_results.scores)
6   for doc in union(dense_results, sparse_results):
7       doc.hybrid_score ← α * norm_dense.get(doc, 0)
8                          + (1-α) * norm_sparse.get(doc, 0)
9   // Option B: Rank-level fusion (RRF)
10  for doc in union(dense_results, sparse_results):
11      doc.rrf_score ← 0
12      if doc in dense_results:
13          doc.rrf_score += α / (κ + dense_rank(doc))
14      if doc in sparse_results:
15          doc.rrf_score += (1-α) / (κ + sparse_rank(doc))
16  hybrid_results ← sort_by_score(all_docs, top_k=K)
17  return hybrid_results
```

### 4.4 Embedding Model Fine-Tuning

When the domain vocabulary and semantic relationships diverge significantly from the embedding model's pretraining distribution, fine-tuning is necessary to realign the embedding space.

**Contrastive fine-tuning** optimizes the embedding model to place semantically similar pairs $(q, d^+)$ closer and dissimilar pairs $(q, d^-)$ farther in the embedding space.

**InfoNCE Loss (SOTA for embedding fine-tuning):**

$$
\mathcal{L}_{\text{InfoNCE}} = -\log \frac{\exp\!\left(\text{sim}(\mathbf{e}_q, \mathbf{e}_{d^+}) / \tau\right)}{\exp\!\left(\text{sim}(\mathbf{e}_q, \mathbf{e}_{d^+}) / \tau\right) + \sum_{i=1}^{N_{\text{neg}}} \exp\!\left(\text{sim}(\mathbf{e}_q, \mathbf{e}_{d_i^-}) / \tau\right)}
$$

where $\tau$ is the temperature hyperparameter controlling the sharpness of the distribution, $d^+$ is a positive (relevant) document, and $\{d_i^-\}$ are negative (irrelevant) documents.

**Hard negative mining** is critical for fine-tuning quality. SOTA approaches use the current model's own retrievals as hard negatives:

$$
d_{\text{hard}}^- = \arg\max_{d \in \mathcal{C} \setminus \{d^+\}} \; \text{sim}(\mathbf{e}_q, \mathbf{e}_d)
$$

These are documents that the model currently considers similar to the query but are actually irrelevant—the most informative training signal.

**Multiple Negatives Ranking Loss (MNRL)** provides efficient in-batch negative sampling:

$$
\mathcal{L}_{\text{MNRL}} = -\frac{1}{B} \sum_{i=1}^{B} \log \frac{\exp\!\left(\text{sim}(\mathbf{e}_{q_i}, \mathbf{e}_{d_i^+}) / \tau\right)}{\sum_{j=1}^{B} \exp\!\left(\text{sim}(\mathbf{e}_{q_i}, \mathbf{e}_{d_j^+}) / \tau\right)}
$$

where $B$ is the batch size and all other positive documents in the batch serve as negatives for each query.

**Matryoshka Representation Learning (MRL)** is a SOTA technique enabling variable-dimension embeddings from a single model. The loss is applied at multiple truncation points simultaneously:

$$
\mathcal{L}_{\text{MRL}} = \sum_{m \in \mathcal{M}} \mathcal{L}_{\text{InfoNCE}}\!\left(\mathbf{e}_q^{[:m]}, \mathbf{e}_d^{[:m]}\right)
$$

where $\mathcal{M} = \{32, 64, 128, 256, 768, \dots\}$ is the set of dimensionality truncation points. This produces embeddings that remain effective at lower dimensions, enabling adaptive cost-accuracy trade-offs at inference time.

```
ALGORITHM 10: EmbeddingFineTuning
─────────────────────────────────────────────────────────────
Input:  base_model M, training_pairs T = [(q, d⁺, {d⁻})],
        val_pairs V, epochs E, τ, lr
Output: fine_tuned_model M*

1   optimizer ← AdamW(M.parameters(), lr=lr)
2   for epoch in 1..E:
3       for batch in shuffle(T):
4           // Hard negative mining from current model
5           for (q, d⁺, negs) in batch:
6               hard_negs ← mine_hard_negatives(M, q, d⁺, corpus, k=N_neg)
7               negs ← negs ∪ hard_negs
8           // Compute InfoNCE loss with MRL
9           loss ← 0
10          for m in MATRYOSHKA_DIMS:
11              e_q ← M.encode(queries, truncate=m)
12              e_pos ← M.encode(positives, truncate=m)
13              e_neg ← M.encode(negatives, truncate=m)
14              loss += InfoNCE(e_q, e_pos, e_neg, τ)
15          loss.backward()
16          optimizer.step()
17      // Validation: measure retrieval quality
18      recall@K ← evaluate_retrieval(M, V)
19      if recall@K < best_recall:
20          patience -= 1
21          if patience == 0: break
22      else:
23          best_recall ← recall@K
24          M* ← M.checkpoint()
25  return M*
```

**Evaluation of fine-tuned embeddings** uses standard retrieval metrics on held-out query-document pairs:

$$
\text{Recall@}K = \frac{|\{d^+ \in \text{top}_K(q)\}|}{|\{d^+\}|}
$$

$$
\text{MRR} = \frac{1}{|\mathcal{Q}|} \sum_{q \in \mathcal{Q}} \frac{1}{\text{rank}(d^+_q)}
$$

$$
\text{NDCG@}K = \frac{\text{DCG@}K}{\text{IDCG@}K}, \quad \text{DCG@}K = \sum_{i=1}^{K} \frac{2^{\text{rel}_i} - 1}{\log_2(i + 1)}
$$

---

## 5. Post-Retrieval Optimization

### 5.1 Re-Ranking

Re-ranking applies a cross-encoder model that jointly encodes the query and each candidate document to produce a refined relevance score:

$$
s_{\text{rerank}}(q, d) = \text{CrossEncoder}([q; d]) \in \mathbb{R}
$$

where $[q; d]$ denotes the concatenation of query and document tokens separated by a special token.

The cross-encoder architecture enables full bidirectional attention between query and document tokens, capturing fine-grained interactions that bi-encoder (embedding) models miss. The computational cost is $O(K \cdot (|q| + |d|)^2)$ per query for $K$ candidates, which is why re-ranking is applied only to a pre-filtered candidate set.

**SOTA re-ranking pipeline:**

$$
\text{top}_K^{\text{initial}} \xrightarrow{\text{vector search}} \text{top}_{K_{\text{over}}} \xrightarrow{\text{cross-encoder}} \text{top}_{K_{\text{final}}}
$$

where $K_{\text{over}} \gg K_{\text{final}}$ (over-retrieve then prune).

The re-ranked score can be combined with the initial retrieval score:

$$
s_{\text{combined}}(q, d) = \lambda \cdot s_{\text{rerank}}(q, d) + (1 - \lambda) \cdot s_{\text{retrieval}}(q, d)
$$

**ColBERT-style late interaction re-ranking** provides a middle ground between bi-encoder speed and cross-encoder accuracy:

$$
s_{\text{ColBERT}}(q, d) = \sum_{i=1}^{|q|} \max_{j \in \{1, \dots, |d|\}} \; \mathbf{q}_i^\top \mathbf{d}_j
$$

where $\mathbf{q}_i$ and $\mathbf{d}_j$ are contextualized token embeddings. Documents can be pre-computed and cached, with only the MaxSim operation at query time.

```
ALGORITHM 11: RetrieveAndRerank
─────────────────────────────────────────────────────────────
Input:  query q, retriever, reranker, K_over, K_final
Output: reranked_context

1   candidates ← retriever.search(q, top_k=K_over)
2   scored_candidates ← []
3   for doc in candidates:
4       rerank_score ← reranker.score(q, doc.text)
5       scored_candidates.append((doc, rerank_score))
6   reranked ← sort(scored_candidates, by=rerank_score, desc=True)
7   reranked_context ← reranked[:K_final]
8   return reranked_context
```

### 5.2 Context Post-Processing

#### 5.2.1 Sentence Window Retrieval (Context Enhancement)

This technique decouples the **retrieval unit** from the **generation unit**. Small chunks (individual sentences) are used for precise retrieval, but a larger context window stored in metadata is substituted before generation:

Let $c_i$ be a sentence-level chunk, and $W(c_i, n)$ be the window of $n$ sentences before and after $c_i$:

$$
W(c_i, n) = (c_{i-n}, \dots, c_{i-1}, c_i, c_{i+1}, \dots, c_{i+n})
$$

At retrieval time, similarity is computed against the sentence embedding $\mathbf{e}_{c_i}$, but the context passed to generation is $W(c_i, n)$.

$$
\text{generation\_context} = \bigcup_{c_i \in \text{top}_K} W(c_i, n)
$$

With deduplication and ordering applied to overlapping windows.

```
ALGORITHM 12: SentenceWindowRetrieval
─────────────────────────────────────────────────────────────
Input:  query q, sentence_index, window_size n, K
Output: enhanced_context

1   sentence_hits ← sentence_index.search(q, top_k=K)
2   windows ← []
3   for hit in sentence_hits:
4       window ← hit.metadata.window  // pre-stored n-sentence window
5       windows.append((hit.score, window))
6   // Deduplicate overlapping windows, merge contiguous
7   merged ← merge_overlapping_windows(windows)
8   // Order by document position for coherent reading
9   enhanced_context ← sort_by_position(merged)
10  return enhanced_context
```

#### 5.2.2 Context Compression

Context compression reduces token count while preserving information critical to answering the query. This directly reduces generation cost and can improve answer quality by removing distracting noise.

**Extractive compression** selects the most relevant sentences/passages:

$$
\mathcal{C}_{\text{compressed}} = \arg\max_{S \subseteq \mathcal{C},\; |S| \leq B_{\text{tok}}} \sum_{s \in S} \text{Relevance}(s, q)
$$

where $B_{\text{tok}}$ is the target token budget.

**Abstractive compression** uses an LLM to summarize retrieved context:

$$
\mathcal{C}_{\text{compressed}} = \text{LLM}_{\text{compress}}\!\left(\mathcal{C},\; q,\; \texttt{prompt}_{\text{compress}}\right)
$$

**SOTA: LLMLingua / LongLLMLingua** applies token-level perplexity-based pruning. For each token $t_i$ in the context, compute its information content under a small reference model $M_{\text{ref}}$:

$$
\text{info}(t_i) = -\log P_{M_{\text{ref}}}(t_i \mid t_{<i})
$$

Tokens with low information content (highly predictable, hence redundant) are pruned:

$$
\mathcal{C}_{\text{compressed}} = \{t_i \in \mathcal{C} : \text{info}(t_i) \geq \theta_{\text{info}}\}
$$

The threshold $\theta_{\text{info}}$ is set to achieve a target compression ratio $\rho = |\mathcal{C}_{\text{compressed}}| / |\mathcal{C}|$.

```
ALGORITHM 13: PerplexityBasedContextCompression
─────────────────────────────────────────────────────────────
Input:  context C, query q, ref_model, target_ratio ρ
Output: compressed_context

1   // Compute query-aware information score per token
2   tokens ← tokenize(C)
3   q_tokens ← tokenize(q)
4   conditioned_input ← q_tokens + tokens
5   logprobs ← ref_model.logprobs(conditioned_input)
6   info_scores ← [-lp for lp in logprobs[len(q_tokens):]]
7   // Determine threshold for target compression ratio
8   target_count ← int(len(tokens) * ρ)
9   θ_info ← sorted(info_scores, reverse=True)[target_count]
10  // Select tokens above threshold
11  compressed_tokens ← [t for t, s in zip(tokens, info_scores) if s ≥ θ_info]
12  compressed_context ← detokenize(compressed_tokens)
13  return compressed_context
```

### 5.3 Prompt Engineering for RAG

Prompt engineering in the RAG context is the compilation of retrieved evidence, system policy, and task directives into a structured prefill that maximizes generation fidelity under a bounded token budget.

#### 5.3.1 Chain-of-Thought (CoT) for RAG

CoT instructs the model to produce intermediate reasoning steps before the final answer:

$$
\mathcal{G}_{\text{CoT}}(q, \mathcal{C}) = \text{LLM}\!\left(\texttt{prompt}_{\text{CoT}},\; q,\; \mathcal{C}\right) \rightarrow (r_1, r_2, \dots, r_T, a)
$$

where $r_i$ are reasoning steps and $a$ is the final answer. This is critical when retrieved documents contain conflicting evidence requiring multi-step resolution.

#### 5.3.2 Tree of Thoughts (ToT) for RAG

ToT extends CoT by exploring multiple reasoning branches:

$$
\text{ToT}(q, \mathcal{C}) = \arg\max_{\text{path} \in \mathcal{T}} \text{Eval}(\text{path})
$$

where $\mathcal{T}$ is a tree of possible reasoning chains and $\text{Eval}$ is a value function (self-evaluation or critic model) that scores each path. This is applied when multiple retrieved documents suggest different answers and the model must systematically evaluate each possibility.

```
ALGORITHM 14: TreeOfThoughtsRAG
─────────────────────────────────────────────────────────────
Input:  query q, context C, generator_llm, evaluator_llm,
        branch_factor b, max_depth D
Output: best_response

1   root ← Node(state=q + C, depth=0)
2   frontier ← PriorityQueue([root])
3   best ← null; best_score ← -∞
4   while frontier not empty:
5       node ← frontier.pop_best()
6       if node.depth ≥ D:
7           score ← evaluator_llm.evaluate(node.reasoning_path, q, C)
8           if score > best_score:
9               best ← node; best_score ← score
10          continue
11      // Generate b candidate next-thoughts
12      thoughts ← generator_llm.generate_thoughts(node.state, n=b)
13      for thought in thoughts:
14          child ← Node(state=node.state + thought,
15                       depth=node.depth + 1,
16                       reasoning_path=node.reasoning_path + [thought])
17          child.priority ← evaluator_llm.score_partial(child.state)
18          frontier.push(child)
19  best_response ← generator_llm.synthesize(best.reasoning_path)
20  return best_response
```

#### 5.3.3 ReAct (Reasoning and Acting) for RAG

ReAct creates a closed-loop between reasoning and action, enabling dynamic retrieval and tool use:

$$
\text{ReAct}: \quad s_t \xrightarrow{\text{Thought}} s_{t+1} \xrightarrow{\text{Action}} o_{t+1} \xrightarrow{\text{Observation}} s_{t+2} \xrightarrow{\text{Thought}} \cdots
$$

Each iteration either retrieves additional evidence, queries a tool, or produces the final answer.

```
ALGORITHM 15: ReActRAGLoop
─────────────────────────────────────────────────────────────
Input:  query q, tools T, retriever, generator_llm, max_steps N
Output: final_answer

1   trajectory ← [(type="query", content=q)]
2   for step in 1..N:
3       response ← generator_llm.react_step(trajectory, tools.schemas())
4       if response.type == "THOUGHT":
5           trajectory.append(("thought", response.content))
6       elif response.type == "ACTION":
7           tool ← T.get(response.tool_name)
8           observation ← tool.execute(response.tool_input, timeout=DEADLINE)
9           trajectory.append(("action", response.content))
10          trajectory.append(("observation", observation))
11      elif response.type == "ANSWER":
12          return response.content
13  // Max steps exceeded — synthesize from available trajectory
14  return generator_llm.forced_answer(trajectory)
```

### 5.4 LLM Fine-Tuning for RAG

Fine-tuning the generative LLM adapts the model to the domain-specific response format, terminology, and reasoning patterns required by the RAG application.

**Supervised Fine-Tuning (SFT)** on domain-specific $(q, \mathcal{C}, a^*)$ triples:

$$
\mathcal{L}_{\text{SFT}} = -\sum_{t=1}^{|a^*|} \log P_\theta(a^*_t \mid q, \mathcal{C}, a^*_{<t})
$$

where the model learns to generate the reference answer $a^*$ conditioned on the query and retrieved context.

**SOTA: Retrieval-Augmented Fine-Tuning (RAFT — Zhang et al., 2024)**

RAFT trains the model to distinguish between relevant ("oracle") and irrelevant ("distractor") retrieved documents during fine-tuning. The training data includes:

- **Oracle documents** $D^*$: the ground-truth relevant documents
- **Distractor documents** $D^-$: irrelevant documents sampled from the retriever

The training mix:

$$
\text{Context}_{\text{RAFT}} = \begin{cases} D^* \cup D^-_{\text{sample}} & \text{with probability } p \\ D^-_{\text{sample}} & \text{with probability } 1 - p \end{cases}
$$

The model learns to identify and ground its answers on the oracle documents while ignoring distractors, and in the second case, to rely on parametric knowledge when no relevant document is present.

**Parameter-Efficient Fine-Tuning (PEFT) with LoRA:**

Instead of updating all model parameters $\theta \in \mathbb{R}^{|\theta|}$, LoRA decomposes weight updates into low-rank matrices:

$$
W' = W + \Delta W = W + BA, \quad B \in \mathbb{R}^{d \times r}, \; A \in \mathbb{R}^{r \times k}, \; r \ll \min(d, k)
$$

where $r$ is the rank hyperparameter. This reduces trainable parameters by a factor of $\frac{d \cdot k}{r \cdot (d + k)}$, enabling fine-tuning on limited hardware.

```
ALGORITHM 16: RAFTFineTuning
─────────────────────────────────────────────────────────────
Input:  base_llm M, training_data T = [(q, D*, a*)],
        corpus C, retriever, oracle_prob p, epochs E
Output: fine_tuned_llm M*

1   training_examples ← []
2   for (q, D*, a*) in T:
3       D⁻ ← retriever.search(q, top_k=K) \ D*  // distractor docs
4       if random() < p:
5           context ← shuffle(D* ∪ sample(D⁻, k=K-|D*|))
6       else:
7           context ← sample(D⁻, k=K)
8       // Generate chain-of-thought answer grounded in oracle docs
9       cot_answer ← format_cot(a*, D*, q)
10      training_examples.append((q, context, cot_answer))
11  // Apply LoRA adapters
12  M_lora ← attach_lora(M, rank=r, target_modules=["q_proj", "v_proj"])
13  for epoch in 1..E:
14      for batch in shuffle(training_examples):
15          loss ← cross_entropy(M_lora(batch.q, batch.context), batch.answer)
16          loss.backward()
17          optimizer.step()
18  M* ← merge_lora(M_lora)
19  return M*
```

---

## 6. End-to-End Evaluation Framework

A production RAG system requires continuous, automated evaluation across every pipeline stage. The evaluation framework must measure component-level and system-level quality.

### 6.1 Retrieval Quality Metrics

**Precision@K:**

$$
\text{Precision@}K = \frac{|\{\text{relevant docs in top-}K\}|}{K}
$$

**Recall@K:**

$$
\text{Recall@}K = \frac{|\{\text{relevant docs in top-}K\}|}{|\{\text{all relevant docs}\}|}
$$

**Mean Reciprocal Rank (MRR):**

$$
\text{MRR} = \frac{1}{|\mathcal{Q}|} \sum_{q \in \mathcal{Q}} \frac{1}{\text{rank}_q(d^+)}
$$

**Normalized Discounted Cumulative Gain (NDCG@K):**

$$
\text{NDCG@}K = \frac{\sum_{i=1}^{K} \frac{2^{\text{rel}_i} - 1}{\log_2(i + 1)}}{\sum_{i=1}^{K} \frac{2^{\text{rel}_i^*} - 1}{\log_2(i + 1)}}
$$

where $\text{rel}_i^*$ is the ideal ranking.

### 6.2 Generation Quality Metrics

**Faithfulness** (fraction of generated claims supported by retrieved context):

$$
\text{Faithfulness}(a, \mathcal{C}) = \frac{|\{\text{claims}(a) \cap \text{supported\_by}(\mathcal{C})\}|}{|\text{claims}(a)|}
$$

**Answer Relevance** (semantic similarity between query and generated answer):

$$
\text{Relevance}(q, a) = \text{sim}\!\left(\mathbf{e}_q, \mathbf{e}_a\right)
$$

**Context Relevance** (fraction of retrieved context that is actually relevant):

$$
\text{ContextRelevance}(q, \mathcal{C}) = \frac{|\{c \in \mathcal{C} : \text{relevant}(c, q)\}|}{|\mathcal{C}|}
$$

**Hallucination Rate** (fraction of generated claims not grounded in context or knowledge):

$$
\text{HallucinationRate} = 1 - \text{Faithfulness}
$$

### 6.3 RAGAS Framework (Automated RAG Evaluation)

The RAGAS score combines multiple dimensions into a single quality metric:

$$
\text{RAGAS} = \text{HarmonicMean}\!\left(\text{Faithfulness},\; \text{AnswerRelevance},\; \text{ContextPrecision},\; \text{ContextRecall}\right)
$$

```
ALGORITHM 17: ContinuousRAGEvaluation
─────────────────────────────────────────────────────────────
Input:  rag_pipeline, eval_dataset E = [(q, a*, relevant_docs)],
        eval_llm, thresholds T
Output: eval_report, pass/fail

1   metrics ← {faithfulness: [], relevance: [], context_precision: [],
2               context_recall: [], latency: [], cost: []}
3   for (q, a*, rel_docs) in E:
4       t_start ← now()
5       retrieved_docs, generated_answer, token_count ← rag_pipeline(q)
6       latency ← now() - t_start
7       // Compute retrieval metrics
8       context_precision ← precision_at_k(retrieved_docs, rel_docs)
9       context_recall ← recall_at_k(retrieved_docs, rel_docs)
10      // Compute generation metrics (LLM-as-judge)
11      faithfulness ← eval_llm.judge_faithfulness(generated_answer, retrieved_docs)
12      relevance ← eval_llm.judge_relevance(q, generated_answer)
13      // Record
14      metrics.faithfulness.append(faithfulness)
15      metrics.relevance.append(relevance)
16      metrics.context_precision.append(context_precision)
17      metrics.context_recall.append(context_recall)
18      metrics.latency.append(latency)
19      metrics.cost.append(token_count * COST_PER_TOKEN)
20  // Aggregate and gate
21  report ← aggregate(metrics)
22  pass ← all(report[m].mean ≥ T[m] for m in T.keys())
23  return report, pass
```

---

## 7. Production Architecture: Unified Pipeline Integration

The complete AURORA pipeline integrates all stages into a typed, observable, fault-tolerant execution graph.

```
ALGORITHM 18: AURORAEndToEndPipeline
─────────────────────────────────────────────────────────────
Input:  user_query q, session_context H, config C
Output: response r, provenance P, metrics M

 1  // ─── PRE-RETRIEVAL ───
 2  t₀ ← now()
 3  routing_decision ← QUERY_ROUTER(q, H, C.route_registry)
 4  transformed_queries ← []
 5  for route in routing_decision.routes:
 6      if route.requires_decomposition:
 7          sub_queries ← DECOMPOSE(q, route)           // Algo 6
 8      else:
 9          sub_queries ← [REWRITE(q, H)]                // Algo 4
10      if route.enable_expansion:
11          sub_queries ← EXPAND(sub_queries)             // Algo 5
12      transformed_queries.append((route, sub_queries))
13
14  // ─── RETRIEVAL ───
15  all_candidates ← []
16  PARALLEL for (route, sqs) in transformed_queries:
17      for sq in sqs:
18          if route.type == "hybrid":
19              hits ← HYBRID_SEARCH(sq, α=route.α, K=C.K_over)  // Algo 9
20          elif route.type == "structured":
21              hits ← SQL_SEARCH(sq, route.schema)
22          elif route.type == "tool":
23              hits ← TOOL_CALL(sq, route.tool, timeout=C.deadline)
24          hits ← APPLY_METADATA_FILTERS(hits, sq.filters)
25          hits ← AUTOCUT(hits, γ=C.γ)                  // Algo 8
26          all_candidates.extend(hits)
27
28  // ─── POST-RETRIEVAL ───
29  deduplicated ← DEDUP_BY_ID(all_candidates)
30  reranked ← RERANK(q, deduplicated, K_final=C.K_final) // Algo 11
31  
32  if C.enable_sentence_window:
33      context ← SENTENCE_WINDOW(reranked, n=C.window_n)  // Algo 12
34  else:
35      context ← reranked
36  
37  if C.enable_compression:
38      context ← COMPRESS(context, q, ρ=C.compression_ratio) // Algo 13
39
40  // ─── PROMPT COMPILATION ───
41  prompt ← COMPILE_PROMPT(
42      role_policy   = C.system_prompt,
43      task          = q,
44      evidence      = context,     // provenance-tagged
45      memory        = SESSION_MEMORY.get(H.session_id),
46      constraints   = C.output_schema,
47      token_budget  = C.max_prompt_tokens
48  )
49
50  // ─── GENERATION ───
51  r ← LLM.generate(prompt,
52                    max_tokens=C.max_gen_tokens,
53                    temperature=C.temperature)
54
55  // ─── VERIFICATION ───
56  faithfulness ← VERIFY_FAITHFULNESS(r, context)
57  if faithfulness < C.faithfulness_threshold:
58      r ← REPAIR_WITH_COT(q, context, r)              // Algo 14/15
59
60  // ─── OBSERVABILITY ───
61  P ← build_provenance(transformed_queries, all_candidates,
62                        reranked, context, r)
63  M ← {latency: now()-t₀, tokens_in: |prompt|, tokens_out: |r|,
64        faithfulness: faithfulness, routes_used: routing_decision}
65  EMIT_TRACE(P, M)
66
67  return r, P, M
```

---

## 8. Optimization Summary Matrix

| Pipeline Stage | SOTA Technique | Key Equation / Mechanism | Primary Metric |
|---|---|---|---|
| **Parsing** | ColPali/ColQwen vision embeddings | $\text{score} = \sum_i \max_j \mathbf{q}_i^\top \mathbf{v}_j$ | Extraction accuracy |
| **Chunking** | Late chunking | $\mathbf{c}_k = \text{mean}(\mathbf{h}_{b_{k-1}+1:b_k})$ | Retrieval precision + context fidelity |
| **Query Rewriting** | Rewrite-Retrieve-Read with RL | $\mathcal{L} = -\mathbb{E}[R(\mathcal{G}(\hat{q}), a^*)]$ | Downstream answer quality |
| **Query Expansion** | RAG-Fusion + RRF | $\text{RRF}(d) = \sum_i \frac{1}{\kappa + \text{rank}_i(d)}$ | Recall@K |
| **Query Decomposition** | Structured decomposition + filter extraction | Sub-query + keyword + metadata predicate | Multi-facet coverage |
| **Query Routing** | Agentic router with tool-use | $\pi(q) = \arg\max_r P(r \mid q, \theta)$ | Route accuracy |
| **Metadata Filtering** | Pre-filter with freshness decay | $\text{freshness} = \exp(-\lambda(t_{\text{now}} - t_d))$ | Precision under time sensitivity |
| **Outlier Exclusion** | Autocut (adaptive cluster truncation) | $i^* = \arg\max_i \Delta_i$ s.t. $\Delta_{i^*} > \mu + \gamma\sigma$ | Noise rejection rate |
| **Hybrid Search** | RRF-fused dense + BM25 | $s = \alpha \hat{s}_{\text{dense}} + (1-\alpha)\hat{s}_{\text{BM25}}$ | NDCG@K |
| **Embedding Fine-Tuning** | InfoNCE + MRL + hard negative mining | $\mathcal{L}_{\text{MRL}} = \sum_m \mathcal{L}_{\text{InfoNCE}}^{[:m]}$ | Recall@K, MRR |
| **Re-Ranking** | Cross-encoder / ColBERT late interaction | $s = \text{CrossEncoder}([q; d])$ | NDCG@K after rerank |
| **Context Enhancement** | Sentence window retrieval | $W(c_i, n) = (c_{i-n}, \dots, c_{i+n})$ | Generation sufficiency |
| **Context Compression** | LLMLingua perplexity-based pruning | $\text{info}(t_i) = -\log P(t_i \mid t_{<i})$ | Compression ratio vs. faithfulness |
| **Prompting** | ReAct + ToT | Thought → Action → Observation loop | Answer correctness on complex queries |
| **LLM Fine-Tuning** | RAFT + LoRA | $W' = W + BA,\; r \ll \min(d,k)$ | Faithfulness, domain accuracy |
| **Evaluation** | RAGAS + CI/CD quality gates | $\text{RAGAS} = \text{HM}(F, R, CP, CR)$ | Regression prevention |

---

## 9. Architectural Invariants

The following invariants must hold across all deployments of the AURORA architecture:

1. **Provenance traceability**: Every generated token must be traceable to its source chunk, retrieval score, route, and transformation chain.
2. **Token budget enforcement**: Total prompt tokens must satisfy $|P| \leq B_{\text{ctx}} - B_{\text{gen}} - B_{\text{safety}}$, where $B_{\text{ctx}}$ is the context window, $B_{\text{gen}}$ is the reserved generation budget, and $B_{\text{safety}}$ is a safety margin.
3. **Idempotent retrieval**: Given identical query, index state, and configuration, the retrieval pipeline must produce identical results.
4. **Bounded recursion**: All agentic loops (ReAct, ToT, decomposition) are bounded by $N_{\text{max}}$ iterations with forced termination and degraded-but-valid output.
5. **Evaluation gating**: No pipeline configuration change is deployed without passing the continuous evaluation suite (Algorithm 17) above all threshold gates.
6. **Cost observability**: Every pipeline execution emits token counts, latency, and estimated cost, enabling real-time budget enforcement and anomaly detection.

---

*AURORA v1.0 — Advanced Unified Retrieval-Optimized Reasoning Architecture. Principal-level technical specification for production-grade retrieval-augmented generation systems.*

### Agentic Retrieval-Augmented Architecture: End-to-End System Design

Modern Retrieval-Augmented Generation (RAG) must be architected as a typed, deterministic protocol stack rather than ad-hoc prompt glue. A production-grade agentic system requires explicit execution boundaries, utilizing JSON-RPC at the user/application layer, gRPC/Protobuf for low-latency internal service execution, and the Model Context Protocol (MCP) for tool discovery. This document defines the principal-level architecture for optimizing the end-to-end retrieval, context engineering, and generation stack.

---

### Phase 1: Deterministic Indexing and Context Representation

Indexing optimization dictates the upper bound of retrieval accuracy. Raw data must be transformed into highly structured, addressable entities before entering the vector manifold.

#### Multimodal Pre-Processing and Parsing
Data pre-processing must preserve semantic lineage. Standard Optical Character Recognition (OCR) is deprecated in favor of vision-language embedding architectures (e.g., ColPali, ColQwen). These multimodal models project document images directly into a high-dimensional latent space, preserving spatial semantics, tabular structures, and typographical hierarchies natively. Text-based processing requires DOM-to-Abstract Syntax Tree (AST) compilation, ensuring that document metadata (lineage, access control lists, timestamps) is explicitly mapped to every parsed node.

#### Chunking Topologies
Chunking dictates the contextual boundary of the retrieval unit. We strictly categorize chunking per document class to optimize downstream synthesis:
*   **Structural & Hierarchical Chunking:** Applied to AST-parsed data (Markdown, HTML). Bounds are defined by semantic tags (e.g., `<h2>` to `<h2>`), preserving internal referential integrity.
*   **Semantic Chunking:** Detects thematic shifts by calculating cosine distance between adjacent sentence embeddings. A new chunk boundary is instantiated when the contextual drift exceeds a statistically derived threshold $\tau$.
*   **Late Chunking (Contextual Embedding):** Processes the entire document through a long-context transformer before extracting chunk-level embeddings via mean pooling of the localized token spans. This ensures each chunk embedding retains the global document context.

Given a document $D$ composed of tokens $t_1, t_2, \dots, t_n$, and a chunk $C_i$ spanning from index $j$ to $k$:
$$ E_{late}(C_i) = \frac{1}{k - j + 1} \sum_{m=j}^{k} h_m $$
where $h_m$ represents the hidden state of token $t_m$ after self-attention over the global document $D$.

---

### Phase 2: Pre-Retrieval Orchestration and Query Compilation

Raw user queries are intrinsically flawed, underspecified, and unsuitable for direct dense retrieval. The pre-retrieval phase treats the query as an execution target, compiling it via semantic transformation and decomposition.

#### Query Transformation: Rewrite and Expand
We model query transformation as a `Rewrite-Retrieve-Read` loop. The orchestration layer invokes a specialized, low-latency Language Model to normalize the query and expand it into $N$ orthogonal semantic vectors.

$$ Q_{expanded} = \{q_1, q_2, \dots, q_N\} \sim P_{\theta}(q | Q_{raw}) $$

This expands the retrieval surface area, counteracting the vocabulary mismatch problem inherent in dense embeddings.

#### Graph-Based Query Decomposition and Routing
Complex inputs are routed through a decomposing agent that maps the query into a Directed Acyclic Graph (DAG) of atomic sub-queries. Sub-queries are concurrently executed across varying execution tiers based on schema and latency requirements. Query routing operates via typed gRPC contracts, directing execution to specific vector indices, relational databases (via Text-to-SQL MCP tools), or deterministic APIs.

```python
# Pseudo-Algorithm: DAG Query Decomposition & Execution
def execute_query_dag(complex_query: str) -> State:
    dag_nodes = DecomposerLLM.parse_to_dag(complex_query)
    futures = {}
    
    for node in topological_sort(dag_nodes):
        if node.dependencies_met(futures):
            # Route via explicit MCP typed contract
            if node.intent == 'semantic':
                futures[node.id] = async_execute(VectorRetriever, node.query)
            elif node.intent == 'factual':
                futures[node.id] = async_execute(SQLTool, node.query)
                
    return await gather_and_synthesize(futures)
```

---

### Phase 3: Provenance-Driven Hybrid Retrieval Engine

Retrieval must function as a deterministic engine, returning provenance-tagged evidence rather than anonymous context blobs.

#### Hybrid Search and Rank Fusion
Relying solely on dense vector retrieval is fragile against exact-match entity queries. We implement hybrid retrieval, combining dense semantic search with sparse lexical search (e.g., BM25). The fusion of these retrieval streams is governed by a convex combination hyperparameter $\alpha \in [0, 1]$ or through Reciprocal Rank Fusion (RRF).

$$ S_{hybrid}(q, d) = \alpha \cdot S_{dense}(E(q), E(d)) + (1 - \alpha) \cdot S_{sparse}(q, d) $$

#### Metadata Filtering and Outlier Exclusion
To enforce hard boundaries on retrieval, we utilize deterministic metadata filtering. Vector indices are pre-filtered via metadata predicates (e.g., RBAC, temporal freshness constraints) before Approximate Nearest Neighbor (ANN) traversal.

To eliminate vector search outliers dynamically, we implement **Autocut** via distance thresholding and density clustering. Instead of a naive $top\_k$, we calculate the first-order derivative of the distance distribution and terminate retrieval at the maximum gradient shift (the "elbow").

```python
# Pseudo-Algorithm: Autocut Outlier Exclusion
def autocut_retrieval(results: List[Result], max_k: int) -> List[Result]:
    if not results: return []
    distances = [r.distance for r in results[:max_k]]
    
    # Calculate step differences between adjacent sorted distances
    deltas = [distances[i+1] - distances[i] for i in range(len(distances)-1)]
    
    # Find the maximum gap (cluster boundary)
    cut_index = deltas.index(max(deltas)) + 1
    return results[:cut_index]
```

#### Contrastive Embedding Fine-Tuning
For specialized domains, base embedding models lack manifold separation. We enforce representation learning via fine-tuning using the InfoNCE or Triplet Margin Loss on domain-specific positive/negative pairs.

$$ \mathcal{L}_{triplet} = \max(0, d(E(q), E(d^+)) - d(E(q), E(d^-)) + \epsilon) $$

This shifts the latent space, tightly clustering domain-specific taxonomy while pushing unrelated concepts past the margin $\epsilon$.

---

### Phase 4: Post-Retrieval Context Compilation

Retrieved assets must be treated as a bounded token budget. We strictly decouple working memory from retrieved context, optimizing for synthesis capacity before generative performance degrades.

#### Cross-Encoder Re-Ranking
Initial hybrid retrieval acts as a high-recall, low-precision bi-encoder phase. We enforce a secondary cross-encoder re-ranking step. The cross-encoder takes the concatenation of the query and document, projecting them through deep self-attention layers to compute a strict relevance score.

$$ score_{relevance} = f_{CE}([CLS] \oplus q \oplus [SEP] \oplus d \oplus [SEP]) $$

#### Context Compression and Sentence Windowing
To maximize token efficiency, redundant data is mechanically pruned. We deploy **Sentence Window Retrieval**: retrieving granular, high-precision semantic boundaries (single sentences), then expanding to the adjacent $k$ sentences utilizing metadata pointers exclusively at prompt compilation time.

Lexical and embedding-based compressors operate on the output, extracting only critical propositions. This establishes a clean active window, preventing attention dilution over "context slop."

---

### Phase 5: Bounded Agentic Loops and Execution Contracts

Prompt engineering is an anti-pattern when treated as stylistic text formatting. We treat "prompts" as compiled runtime artifacts—deterministic prefills assembling role policy, task state, memory summaries, retrieved evidence, and tool affordances.

#### ReAct, ToT, and Bounded State Machines
Agentic execution must operate within a bounded control system, adhering to strict idempotency and recursion limits. Frameworks like Chain of Thought (CoT), Tree of Thoughts (ToT), and Reasoning and Acting (ReAct) are formalized as explicit state machine transitions: `Plan` $\rightarrow$ `Retrieve` $\rightarrow$ `Act` $\rightarrow$ `Verify` $\rightarrow$ `Commit`.

```python
# Pseudo-Algorithm: Bounded Agent Control System
def agent_execution_loop(task: Task, max_steps: int = 5) -> Result:
    state = State(working_memory=[], task=task)
    
    for step in range(max_steps):
        plan = Synthesizer.plan(state)
        
        if plan.is_complete:
            return verify_and_commit(state)
            
        action = ToolRouter.route(plan.next_action) # MCP Typed Contract
        observation = execute_with_backpressure(action)
        
        state.update_working_memory(action, observation)
        state.compress_stale_history() # Enforce token budget
        
    raise RecursionError("Max bounds exceeded. Initiating graceful degradation.")
```

#### Memory Stratification
The agent loop interacts with a heavily stratified memory hierarchy:
1.  **Working Context:** Ephemeral token budget for the current DAG node execution.
2.  **Episodic Memory:** Validated interactions, appended only after passing automated eval gates and deduplication.
3.  **Semantic/Organizational Memory:** Global vector store of domain truths.

Writes to durable memory require explicit provenance capture and expiry policy evaluation.

#### LLM Fine-Tuning for Protocol Adherence
To ensure the LLM strictly complies with structured tool protocols (JSON-RPC) and agentic state transitions without hallucinating schema properties, the generation model itself is fine-tuned. Direct Preference Optimization (DPO) and Low-Rank Adaptation (LoRA) are applied over telemetry derived from CI/CD traces, penalizing verbose conversational padding and rewarding strictly typed, deterministic outputs.

By enforcing rigid architectural boundaries, observability pipelines, and mechanical context constraints, the RAG paradigm transitions from brittle heuristic scripts into a scalable, fault-tolerant enterprise agentic platform.

# Deterministic High-Fidelity RAG
## A Production-Grade Technical Report on Indexing, Pre-Retrieval, Retrieval, and Post-Retrieval Optimization

---

## 1. Problem Definition and End-to-End Objective

A retrieval-augmented generation system is not a single retrieval call followed by prompt stuffing. It is a multi-stage decision system whose objective is to maximize grounded answer quality under strict latency, cost, and reliability constraints.

Let the corpus be $$\mathcal{D} = \{d_1, d_2, \dots, d_n\}$$, the user query be $$q$$, the retrieved evidence set be $$E_q \subset \mathcal{D}$$, and the generated answer be $$a$$.

The production objective is:

$$
\Pi^* = \arg\max_{\Pi} \ \mathbb{E}\Big[
\lambda_1 \cdot \mathrm{Correctness}(a)
+ \lambda_2 \cdot \mathrm{Faithfulness}(a, E_q)
+ \lambda_3 \cdot \mathrm{Coverage}(E_q)
- \lambda_4 \cdot \mathrm{Latency}(\Pi)
- \lambda_5 \cdot \mathrm{Cost}(\Pi)
- \lambda_6 \cdot \mathrm{HallucinationRisk}(a)
\Big]
$$

A naive RAG pipeline implements:

$$
E_q = \operatorname{topk}_{d \in \mathcal{D}} \ \mathrm{sim}(f_{\text{emb}}(q), f_{\text{emb}}(d))
$$

$$
a = f_{\text{LLM}}(q, E_q)
$$

This baseline fails systematically because it ignores:

1. **Document structure**
2. **Query ambiguity**
3. **Lexical exact-match requirements**
4. **Freshness and authority**
5. **Chunk boundary quality**
6. **Context redundancy and prompt inefficiency**
7. **Verification and abstention controls**

A production-grade RAG stack must therefore be decomposed into four optimization layers:

1. **Indexing**
2. **Pre-retrieval**
3. **Retrieval**
4. **Post-retrieval**

---

## 2. Canonical Typed Artifacts

A high-quality RAG system requires explicit typed artifacts with provenance, lineage, and schema versioning.

```ts
type Document = {
  document_id: string
  source_id: string
  source_uri: string
  modality: "text" | "pdf" | "html" | "image" | "spreadsheet" | "code"
  raw_checksum: string
  schema_version: string
  created_at: string
  updated_at: string
  metadata: Record<string, string | number | boolean>
}

type Partition = {
  partition_id: string
  document_id: string
  partition_type: "section" | "table" | "paragraph" | "list" | "cell_range" | "code_block"
  text: string
  span: { start: number, end: number }
  structure_path: string[]
  metadata: Record<string, unknown>
}

type Chunk = {
  chunk_id: string
  document_id: string
  partition_id: string
  text: string
  token_count: number
  embedding_dense?: number[]
  embedding_sparse?: Record<string, number>
  parent_context_id?: string
  window_context?: string
  metadata: {
    title?: string
    headings?: string[]
    author?: string
    timestamp?: string
    language?: string
    access_scope?: string
    doc_type?: string
    authority_score?: number
    freshness_score?: number
  }
  provenance: {
    source_uri: string
    page?: number
    offsets?: { start: number, end: number }
  }
  schema_version: string
}
```

Critical invariants:

1. Every chunk must be **traceable** to source offsets.
2. Every embedding must be tied to **embedding model version**.
3. Every index write must be **idempotent**.
4. Every retrieval candidate must expose **provenance**.
5. Every answer must be reconstructible from **retrieval trace + prompt artifact + model version**.

---

## 3. RAG Pipeline Overview: From Naive Retrieval to Deterministic Evidence Construction

The effective pipeline is:

1. **Documents**
2. **Parsing and normalization**
3. **Partitioning**
4. **Chunking**
5. **Embedding and indexing**
6. **Query transformation**
7. **Query decomposition**
8. **Query routing**
9. **Filtered and hybrid retrieval**
10. **Outlier rejection**
11. **Re-ranking**
12. **Context enhancement and compression**
13. **Prompt compilation**
14. **LLM generation**
15. **Validation and grounded response emission**

The architectural principle is not “retrieve more and hope.” It is:

**retrieve precisely, rank aggressively, compress deterministically, and generate only from provenance-tagged evidence**

---

# 4. Indexing Optimization

## 4.1 Objective

Indexing defines the retrievable information geometry of the system. Poor indexing cannot be repaired downstream.

The indexing objective is:

$$
\mathcal{I}^* = \arg\max_{\mathcal{I}}
\Big[
\mathrm{Retrievability}(\mathcal{I})
+ \mathrm{ContextCompleteness}(\mathcal{I})
- \mathrm{Noise}(\mathcal{I})
- \mathrm{StorageCost}(\mathcal{I})
\Big]
$$

The indexing layer contains two critical subproblems:

1. **Data pre-processing**
2. **Chunking**

---

## 4.2 Data Acquisition and Integration

A robust knowledge base must ingest heterogeneous sources without losing structure.

### Source classes

1. Markdown and plain text
2. PDFs
3. HTML and web applications
4. Office documents
5. Spreadsheets
6. Source code repositories
7. Scanned documents and images
8. Databases and operational systems

### Production requirements

1. **Deduplicate by content checksum**
2. **Version by source revision**
3. **Retain lineage**
4. **Track parser confidence**
5. **Preserve access-control metadata**
6. **Separate raw artifacts from normalized artifacts**

A recommended ingestion record contains:

```ts
type IngestionRecord = {
  source_id: string
  revision_id: string
  acquisition_time: string
  checksum: string
  parser_name: string
  parser_version: string
  extraction_confidence: number
  normalized_schema_version: string
}
```

---

## 4.3 Data Extraction and Parsing

### 4.3.1 Textual formats

For Markdown, Word, and plain text, extraction should preserve:

1. Heading hierarchy
2. Paragraph boundaries
3. Lists
4. Tables
5. Footnotes
6. Inline references
7. Captions
8. Code blocks

### 4.3.2 PDFs and scanned documents

PDF ingestion is not a text-extraction problem only; it is a **layout reconstruction problem**.

Recommended stack by modality:

1. **Digitally native PDFs**: layout-aware text extraction
2. **Scanned PDFs/images**: OCR plus layout recovery
3. **Multimodal retrieval-heavy environments**: direct vision-language retrieval using models such as **ColPali** or **ColQwen**

For scanned financial, scientific, or legal material with dense tables and formulas, direct image-based retrieval may outperform OCR pipelines because OCR often destroys:

1. table topology
2. formula layout
3. multi-column reading order
4. figure-caption attachment

### 4.3.3 HTML parsing

HTML must be parsed structurally, not flattened blindly. Preserve:

1. DOM hierarchy
2. anchor text
3. tables
4. list nesting
5. semantic tags
6. article body vs navigation vs boilerplate

A DOM-aware extractor should identify content blocks by visual density, semantic tags, and template repetition.

### 4.3.4 Spreadsheets

Spreadsheet parsing must preserve:

1. sheet name
2. table boundaries
3. merged-cell semantics
4. header rows
5. row/column relationships
6. formulas and computed values

Chunking spreadsheets as plain row text is usually suboptimal. Better representations are:

1. relational records
2. cell-range blocks
3. table snapshots with headers

### 4.3.5 Metadata extraction

Metadata is a first-class retrieval primitive. Extract:

1. author
2. timestamp
3. source
4. document type
5. topic/domain
6. language
7. security/access scope
8. revision
9. section headings
10. authority score
11. freshness score

---

## 4.4 Data Cleaning and Noise Reduction

Cleaning must improve retrievability without distorting meaning.

### Remove or suppress

1. headers and footers
2. cookie notices
3. navigation chrome
4. page numbers
5. repeated boilerplate
6. disclaimer blocks
7. OCR garbage
8. duplicate paragraphs
9. broken Unicode and encoding artifacts

### Preserve explicitly

1. headings
2. citations
3. table headers
4. figure captions
5. code syntax
6. units
7. dates
8. negation markers

A common failure is over-aggressive cleaning that removes exactly the discriminative features needed for retrieval.

### Cleaning objective

$$
\mathcal{L}_{\text{clean}} =
\lambda_1 \cdot \mathrm{BoilerplateRemoved}
- \lambda_2 \cdot \mathrm{MeaningLost}
- \lambda_3 \cdot \mathrm{StructureLost}
$$

---

## 4.5 Data Transformation and Canonical Schema

Document partitioning is distinct from chunking.

### Partitioning

Partitioning converts a document into **logical elements**:

1. section
2. subsection
3. paragraph
4. table
5. list
6. code block
7. cell range
8. figure caption

### Chunking

Chunking combines one or more partitions into retrieval units optimized for search and downstream generation.

The recommended sequence is:

1. Acquire raw data
2. Parse modality-specific structure
3. Clean noise
4. Normalize to canonical schema
5. Partition into logical elements
6. Chunk using document-class policy
7. Enrich with metadata
8. Embed and index

---

## 4.6 Chunking Strategies

Chunking is the dominant determinant of retrieval precision, answer completeness, and prompt efficiency.

### 4.6.1 Chunking objective

Let chunk set be $$C = \{c_1, c_2, \dots, c_m\}$$. The ideal chunking policy maximizes:

$$
\mathcal{J}_{\text{chunk}} =
\lambda_1 \cdot \mathrm{SemanticCoherence}(C)
+ \lambda_2 \cdot \mathrm{AnswerContainment}(C)
- \lambda_3 \cdot \mathrm{BoundaryLoss}(C)
- \lambda_4 \cdot \mathrm{Redundancy}(C)
- \lambda_5 \cdot \mathrm{TokenWaste}(C)
$$

### 4.6.2 Fixed-size chunking

This is only an acceptable baseline for homogeneous corpora with weak structure.

Advantages:
1. deterministic
2. cheap
3. easy to parallelize

Limitations:
1. breaks semantic units
2. mixes unrelated spans
3. causes boundary fragmentation
4. increases downstream reranking burden

Overlap can reduce boundary loss but increases redundancy.

### 4.6.3 Recursive chunking

Recursive chunking respects structure by trying separators in order:

1. section
2. paragraph
3. sentence
4. token window

This is a strong default for mixed textual corpora.

### 4.6.4 Document-based chunking

For HTML, Markdown, legal agreements, API docs, and code, chunk at natural structural boundaries:

1. heading blocks
2. function/class units
3. section clauses
4. table regions

This preserves semantic boundaries and provenance clarity.

### 4.6.5 Semantic chunking

Semantic chunking detects topic shifts using embedding distances between adjacent units.

Let the sentence embeddings be $$e_1, e_2, \dots, e_T$$. Define adjacent semantic drift:

$$
\Delta_i = 1 - \cos(e_i, e_{i+1})
$$

Split when:

$$
\Delta_i > \mu_{\Delta} + \kappa \sigma_{\Delta}
$$

where $$\kappa$$ is calibrated on validation data.

This is superior to purely length-based segmentation when documents contain topic transitions that do not align cleanly with paragraph boundaries.

### 4.6.6 LLM-based chunking

LLM-based chunking extracts propositions or semantically atomic units, then groups them into coherent retrieval chunks.

Recommended only when:
1. corpus value is high
2. answer precision matters materially
3. indexing can be performed offline
4. latency and cost are acceptable

This is highly effective for:
1. scientific documents
2. legal corpora
3. policy manuals
4. biomedical knowledge bases

### 4.6.7 Late chunking

Late chunking computes long-context embeddings before splitting, so local chunk vectors retain global document context.

This is especially useful when:
1. sections reference distant definitions
2. abbreviations are globally introduced
3. local passages depend heavily on long-range context

### 4.6.8 SOTA recommendation by corpus class

| Corpus class | Recommended chunking policy | Avoid |
|---|---|---|
| API docs / Markdown / HTML | document-based + recursive fallback | pure fixed windows |
| Legal / contracts | section-aware + semantic + parent-child links | small flat chunks |
| Scientific PDFs | layout-aware partitioning + semantic or proposition chunking | OCR-only flat text |
| Source code | AST/function/class chunking | sentence chunking |
| FAQs / short KB articles | recursive chunking with light overlap | overly complex LLM chunking |
| Multi-hop enterprise wiki | semantic chunking + late chunking + window metadata | naive paragraph splits |

---

## 4.7 Pseudo-Algorithm: Adaptive Indexing and Chunking

```text
Algorithm 1: BUILD_TYPED_INDEX
Input:
  raw_documents R
  parser_registry P
  chunk_policy_registry C
  embedder_dense E_d
  embedder_sparse E_s
Output:
  indexed_chunks I

for each raw_document r in R do
    if EXISTS_BY_CHECKSUM(r.checksum, r.source_id, r.revision_id) then
        continue   // idempotent ingestion
    end if

    modality <- DETECT_MODALITY(r)
    parser <- SELECT_PARSER(P, modality, r.mime_type, r.layout_signature)

    parsed <- parser.EXTRACT(r)
    cleaned <- CLEAN_AND_DEDUP(parsed)
    partitions <- PARTITION_TO_CANONICAL_SCHEMA(cleaned)

    doc_class <- CLASSIFY_DOCUMENT_TYPE(partitions, metadata=parsed.metadata)
    policy <- SELECT_CHUNK_POLICY(C, doc_class, structure=partitions.structure_profile)

    chunks <- APPLY_CHUNK_POLICY(policy, partitions)

    for each chunk c in chunks do
        c.metadata <- ENRICH_METADATA(c, parsed.metadata, headings, timestamps, authority)
        c.provenance <- ATTACH_PROVENANCE(r.source_uri, c.offsets, parser.version)
        c.chunk_id <- HASH(r.document_id, c.offsets, c.text, policy.version)

        c.embedding_dense <- E_d(c.text or c.rendered_layout)
        c.embedding_sparse <- E_s(c.text)

        UPSERT_INDEX(c.chunk_id, c)  // idempotent upsert
    end for
end for

return INDEX_HANDLE()
```

---

# 5. Pre-Retrieval Optimization

## 5.1 Objective

Using the raw user query directly for retrieval is often suboptimal because user language is underspecified, noisy, or misaligned with indexed vocabulary.

Pre-retrieval optimizes the search intent before hitting the retrieval substrate.

The objective is:

$$
q^* = \arg\max_{q'} \mathrm{Retrievability}(q')
\quad \text{subject to} \quad
\mathrm{IntentPreservation}(q, q') \ge \tau
$$

Core methods:

1. query transformation
2. query decomposition
3. query routing

---

## 5.2 Query Transformation

### 5.2.1 Query rewriting

Query rewriting reformulates the user query into a retrieval-optimized query.

A retrieval rewrite must perform:

1. entity normalization
2. acronym expansion
3. ambiguity reduction
4. temporal clarification
5. removal of conversational filler
6. conversion to corpus-native terminology

Example transformation class:
- from user phrasing
- to high-recall, schema-compatible search phrasing

The key failure mode is **query drift**, where the rewrite changes the user’s intent.

A rewrite policy should therefore constrain the transformation to preserve:

1. named entities
2. temporal bounds
3. polarity
4. comparison semantics
5. jurisdiction/domain scope

### 5.2.2 Query expansion

Query expansion generates multiple semantically related retrieval queries.

Let expanded set be $$Q^+ = \{q_1, q_2, \dots, q_m\}$$. Retrieval then becomes:

$$
E_q = \bigcup_{i=1}^{m} \operatorname{Retrieve}(q_i)
$$

Expansion is valuable when:
1. domain terminology is inconsistent
2. queries are broad
3. multiple paraphrases map to different evidence clusters

However, expansion increases:
1. candidate count
2. latency
3. duplicate retrieval
4. reranking cost

Therefore expansion must be bounded by:
1. diversity threshold
2. marginal recall gain
3. latency budget

### 5.2.3 SOTA recommendation

Do not use a generic unconstrained rewrite prompt. Use a **structured retrieval query compiler** that outputs:

```ts
type RetrievalQuery = {
  rewritten_query: string
  required_entities: string[]
  optional_synonyms: string[]
  date_constraints?: { from?: string, to?: string }
  metadata_filters?: Record<string, unknown>
  expansion_queries: string[]
  confidence: number
}
```

---

## 5.3 Query Decomposition

Complex queries often conflate multiple latent questions. Retrieval quality improves when these are decomposed into smaller subqueries.

Given a complex query $$Q$$, find:

$$
\{q_1, q_2, \dots, q_k\}
=
\arg\max
\Big[
\mathrm{Coverage}(Q, \{q_i\})
- \lambda \cdot \mathrm{Redundancy}(\{q_i\})
\Big]
$$

Decomposition is especially beneficial for:

1. comparative queries
2. multi-hop queries
3. causal queries
4. temporal queries
5. multi-constraint diagnostic questions

### Decomposition policy

A good decomposition model produces:

1. atomic subqueries
2. dependency graph among subqueries
3. aggregation plan
4. metadata filters per subquery

Example decomposition classes:
1. cause analysis
2. comparison
3. temporal sequence
4. evidence gathering
5. alternative hypothesis evaluation

### Parallelism

Independent subqueries should be retrieved in parallel, but synthesis must retain provenance per subquery.

---

## 5.4 Query Routing

Routing decides which retrieval path or tool chain should execute.

Let route set be $$\mathcal{R}$$. Select:

$$
r^* = \arg\max_{r \in \mathcal{R}} P(r \mid q, m, b)
$$

where:
1. $$q$$ is query content
2. $$m$$ is metadata or user/session context
3. $$b$$ is latency/cost budget

### Typical route classes

1. dense semantic retrieval
2. lexical retrieval
3. hybrid retrieval
4. multimodal retrieval
5. structured query engine
6. web retrieval
7. no-retrieval generation
8. human clarification request

### Routing signals

1. domain
2. query length
3. presence of exact entities
4. temporal freshness requirement
5. modality requirement
6. access scope
7. ambiguity score
8. decomposition complexity
9. expected answer type

### Multi-index strategy

Production RAG should maintain multiple indexes, for example:

1. document-level dense index
2. chunk-level dense index
3. sparse lexical index
4. late-interaction index
5. multimodal document image index
6. table-specific index
7. code index

Routing is then an inference problem over these indexes, not a static configuration choice.

---

## 5.5 Pseudo-Algorithm: Query Rewrite, Decompose, Route

```text
Algorithm 2: PRE_RETRIEVAL_PLAN
Input:
  user_query q
  user_context u
  latency_budget B
Output:
  retrieval_plan P

q_norm <- NORMALIZE_QUERY(q)
complexity <- ESTIMATE_COMPLEXITY(q_norm)
freshness_need <- DETECT_FRESHNESS_REQUIREMENT(q_norm)
entity_set <- EXTRACT_ENTITIES(q_norm)
candidate_filters <- EXTRACT_METADATA_FILTERS(q_norm, u)

rewrite <- RETRIEVAL_REWRITE(
    q_norm,
    preserve_entities=entity_set,
    preserve_polarity=true,
    preserve_time=true
)

if complexity > TAU_DECOMPOSE then
    subqueries <- DECOMPOSE_QUERY(rewrite.rewritten_query)
else
    subqueries <- [rewrite.rewritten_query]
end if

for each sq in subqueries do
    route <- ROUTE_QUERY(
        sq,
        entities=entity_set,
        filters=candidate_filters,
        freshness=freshness_need,
        latency_budget=B
    )
    P.add({
        subquery: sq,
        route: route,
        filters: candidate_filters,
        expansions: DIVERSITY_BOUNDED_EXPANSION(sq, max_expansions=K)
    })
end for

return P
```

---

# 6. Retrieval Optimization

## 6.1 Objective

Retrieval should return an evidence set with maximum answer utility, not merely nearest vectors.

A principled objective is:

$$
E_q^* = \arg\max_{E \subseteq \mathcal{D}}
\Big[
\mathrm{Relevance}(E, q)
+ \mathrm{Authority}(E)
+ \mathrm{Freshness}(E)
+ \mathrm{Coverage}(E, q)
- \mathrm{Noise}(E)
\Big]
$$

Core mechanisms:

1. metadata filtering
2. outlier exclusion
3. hybrid search
4. embedding model fine-tuning

---

## 6.2 Metadata Filtering

Metadata filtering narrows search to candidate subsets that are semantically plausible and operationally valid.

Define a metadata predicate $$\phi(d, q)$$. Candidate set becomes:

$$
\mathcal{D}_q^{\text{filtered}} = \{ d \in \mathcal{D} \mid \phi(d, q) = 1 \}
$$

Typical predicates:

1. tenant or user ownership
2. access scope
3. language
4. document type
5. product/version
6. date range
7. author or authority class
8. source system
9. topic label
10. jurisdiction

### Why metadata filtering matters

Dense retrieval alone cannot encode hard constraints reliably. A semantically similar chunk from the wrong tenant, outdated software version, or irrelevant document class is still wrong.

### Freshness-aware retrieval

If timestamp metadata exists, incorporate recency score:

$$
s_{\text{fresh}}(d) = \exp(-\gamma \cdot \Delta t_d)
$$

A composite retrieval score can be:

$$
s(d, q) =
w_1 s_{\text{dense}}(d, q)
+ w_2 s_{\text{lex}}(d, q)
+ w_3 s_{\text{authority}}(d)
+ w_4 s_{\text{fresh}}(d)
$$

### Design rule

Metadata should be:

1. query-relevant
2. stable
3. typed
4. cheap to filter
5. aligned to business constraints

Do not pollute metadata with low-signal attributes that inflate index complexity without improving retrieval.

---

## 6.3 Excluding Vector Search Outliers

A naive `top_k` search returns the closest points whether they are meaningful or not.

### 6.3.1 Distance thresholding

If vector distance is $$\delta(d, q)$$, retain only:

$$
E_q = \{ d \in \operatorname{topk}(q) \mid \delta(d, q) \le \tau \}
$$

This removes obvious poor matches. However, threshold $$\tau$$ must be calibrated per:

1. embedding model
2. corpus
3. distance metric
4. collection type

A single global threshold across domains is usually incorrect.

### 6.3.2 Autocut

Sort candidates by increasing distance:

$$
\delta_{(1)} \le \delta_{(2)} \le \cdots \le \delta_{(n)}
$$

Find a large gap:

$$
k^* = \arg\max_{i < n} \left( \delta_{(i+1)} - \delta_{(i)} \right)
$$

Then keep only:

$$
E_q = \{ d_{(1)}, d_{(2)}, \dots, d_{(k^*)} \}
$$

This dynamic cutoff is superior when relevant candidates form a dense cluster and outliers lie beyond a distance jump.

### 6.3.3 Recommended practice

Use:
1. over-retrieval
2. metadata filtering
3. distance thresholding
4. autocut
5. reranking

in that order.

---

## 6.4 Hybrid Search

Hybrid retrieval combines lexical precision with semantic generalization.

### 6.4.1 Basic weighted fusion

Let dense and lexical scores be normalized to comparable scales.

$$
s_{\text{hybrid}}(d, q)
=
\alpha \cdot \tilde{s}_{\text{dense}}(d, q)
+
(1-\alpha) \cdot \tilde{s}_{\text{lex}}(d, q)
$$

Interpretation:
1. $$\alpha = 1$$: pure semantic
2. $$\alpha = 0$$: pure lexical
3. intermediate: mixed retrieval

### 6.4.2 Why pure dense retrieval is insufficient

Pure dense retrieval underperforms when the query depends on:

1. exact identifiers
2. product names
3. formulae
4. legal clauses
5. rare domain acronyms
6. error codes
7. numeric literals

### 6.4.3 Why pure lexical retrieval is insufficient

Pure lexical retrieval underperforms when the query uses paraphrase, abstraction, or concept-level similarity.

### 6.4.4 SOTA fusion methods

Weighted linear blending is only a baseline. Stronger production choices are:

1. **Reciprocal Rank Fusion**
2. **learned rank fusion**
3. **tri-modal fusion** using dense + sparse + late interaction

Reciprocal Rank Fusion:

$$
\mathrm{RRF}(d) = \sum_{m \in \mathcal{M}} \frac{1}{k + r_m(d)}
$$

where:
1. $$\mathcal{M}$$ is the set of retrieval methods
2. $$r_m(d)$$ is the rank of document $$d$$ under method $$m$$
3. $$k$$ is a smoothing constant

RRF is robust because it avoids brittle score calibration across heterogeneous retrievers.

### 6.4.5 Recommended retrieval stack

For high-quality enterprise RAG:

1. sparse retriever for exact-match recall
2. dense bi-encoder for semantic recall
3. optional late-interaction retriever for fine-grained passage relevance
4. fusion layer
5. reranker

---

## 6.5 Embedding Model Fine-Tuning

Off-the-shelf embeddings degrade on specialized corpora because domain semantics are not adequately represented.

### 6.5.1 When fine-tuning is justified

Fine-tune embeddings when the corpus contains:

1. high domain specificity
2. uncommon terminology
3. domain-specific paraphrase patterns
4. task-specific notion of relevance
5. critical precision requirements

Examples:
1. medicine
2. law
3. compliance
4. semiconductors
5. scientific literature
6. proprietary enterprise jargon

### 6.5.2 Training data construction

Construct triplets or grouped examples:

1. query
2. positive passage
3. hard negatives

Hard negatives should come from:
1. lexical confusion
2. semantic near misses
3. same-entity wrong-context passages
4. same-topic contradictory passages

### 6.5.3 Contrastive training objective

For query embedding $$e_q$$, positive $$e_+$$, negatives $$e_j^-$$:

$$
\mathcal{L}_{\text{InfoNCE}}
=
-\log
\frac{
\exp(\mathrm{sim}(e_q, e_+) / \tau)
}{
\exp(\mathrm{sim}(e_q, e_+) / \tau)
+
\sum_{j=1}^{N}
\exp(\mathrm{sim}(e_q, e_j^-) / \tau)
}
$$

This pulls relevant pairs together and pushes hard negatives away.

### 6.5.4 SOTA refinements

1. mined hard negatives with cross-encoder teacher
2. instruction-tuned embeddings
3. multi-vector embeddings for long documents
4. matryoshka representation learning for multi-resolution retrieval
5. domain-adaptive continued pretraining before contrastive tuning

### 6.5.5 Evaluation

Evaluate fine-tuned embeddings on held-out retrieval tasks:

1. Recall@k
2. MRR
3. nDCG
4. answer-support coverage
5. robustness across query reformulations

Do not rely only on intrinsic embedding clustering visualizations.

---

## 6.6 Pseudo-Algorithm: Hybrid Retrieval with Filtering and Outlier Rejection

```text
Algorithm 3: RETRIEVE_CANDIDATES
Input:
  retrieval_plan P
  dense_index I_d
  sparse_index I_s
  optional late_index I_l
Output:
  candidate_set C

C <- empty set

for each plan_item p in P do
    for each query_variant qv in [p.subquery] + p.expansions do
        dense_hits <- I_d.SEARCH(qv, filters=p.filters, top_n=N_dense)
        sparse_hits <- I_s.SEARCH(qv, filters=p.filters, top_n=N_sparse)

        if p.route.requires_late_interaction then
            late_hits <- I_l.SEARCH(qv, filters=p.filters, top_n=N_late)
        else
            late_hits <- []
        end if

        fused <- FUSE_RESULTS(dense_hits, sparse_hits, late_hits, method="RRF")

        pruned <- DISTANCE_THRESHOLD(fused, calibrated_thresholds)
        pruned <- AUTOCUT(pruned, min_keep=K_min)

        C <- UNION_WITH_DEDUP(C, pruned)
    end for
end for

return C
```

---

# 7. Post-Retrieval Optimization

## 7.1 Objective

Post-retrieval is where retrieved candidates are converted into a compact, high-utility evidence set for generation.

The objective is:

$$
E_q^{\text{final}} =
\arg\max_{E' \subseteq E_q}
\Big[
\mathrm{AnswerUtility}(E', q)
- \lambda_1 \cdot \mathrm{Redundancy}(E')
- \lambda_2 \cdot \mathrm{TokenCost}(E')
\Big]
$$

Core components:

1. re-ranking
2. context enhancement
3. context compression
4. prompt engineering
5. LLM fine-tuning

---

## 7.2 Re-Ranking

### 7.2.1 Why reranking works

First-stage retrieval is fast because query and documents are encoded separately. It sacrifices cross-token interaction fidelity. Rerankers process query and candidate jointly, allowing fine-grained relevance scoring.

Let reranker score be:

$$
s_{\text{rr}}(d, q) = f_{\theta}(q, d)
$$

where $$f_{\theta}$$ may be:
1. cross-encoder
2. sequence-to-sequence reranker
3. late-interaction verifier

### 7.2.2 Production pattern

1. retrieve broadly: usually 20 to 200 candidates depending on corpus
2. rerank with expensive model
3. keep top 5 to 20 for context assembly

### 7.2.3 Diversity-aware reranking

To prevent near-duplicate chunks from consuming prompt budget, use Maximal Marginal Relevance:

$$
\mathrm{MMR}(d)
=
\lambda \cdot s_{\text{rr}}(d, q)
-
(1-\lambda) \cdot \max_{d' \in S} \mathrm{sim}(d, d')
$$

This balances relevance and diversity.

### 7.2.4 SOTA reranking choices

1. cross-encoders for highest precision
2. late interaction for improved scalability
3. listwise reranking when candidate interdependence matters

### 7.2.5 Failure modes

1. reranker drift on domain terminology
2. latency blowup
3. duplicate heavy top ranks
4. context-window waste

Mitigations:
1. domain-tune reranker
2. cap candidate depth
3. MMR or duplicate suppression
4. cache reranked results for repeated queries

---

## 7.3 Context Post-Processing

### 7.3.1 Context enhancement with metadata

Retrieved chunks should often be enriched with metadata before generation. Valuable metadata includes:

1. title
2. section heading
3. timestamp
4. author
5. source type
6. document version
7. sibling context
8. parent section summary

This is not cosmetic. Metadata helps the generator evaluate:
1. authority
2. chronology
3. topical frame
4. document intent

### 7.3.2 Sentence window retrieval

This is a highly effective precision-recall tradeoff strategy.

**Index small units**, often sentences or short propositions, but attach a larger surrounding window as metadata.

At retrieval time:
1. retrieve the small unit for precision
2. expand to the stored context window for generation quality

Let retrieved unit be $$s_i$$ and context window size be $$w$$. The generation context becomes:

$$
W_i = \{s_{i-w}, \dots, s_i, \dots, s_{i+w}\}
$$

This decouples retrieval granularity from generation granularity.

### 7.3.3 Parent-child retrieval

A related improvement is to retrieve child chunks but inject parent section context only for the selected top candidates. This preserves precision without destroying context completeness.

---

## 7.4 Context Compression

Context compression removes irrelevant or redundant information from retrieved evidence.

### 7.4.1 Compression objective

Given retrieved context set $$E_q$$ and token budget $$B$$, find:

$$
E_q^{\text{compressed}}
=
\arg\max_{E' \subseteq E_q}
\Big[
\mathrm{Support}(q, E')
- \lambda \cdot \mathrm{Redundancy}(E')
\Big]
\quad
\text{subject to}
\quad
\mathrm{Tokens}(E') \le B
$$

### 7.4.2 Compression modes

1. **lexical compression**  
   Keep query-overlapping or high-TF-IDF spans.

2. **embedding-based compression**  
   Retain sentences with highest semantic relevance to the query.

3. **reranker-guided sentence extraction**  
   Score sentence-level relevance after passage retrieval.

4. **generative compression**  
   LLM extracts claim-supporting spans or structured summaries with citations.

### 7.4.3 Recommended production design

Use a deterministic multi-step compressor:

1. deduplicate near-identical chunks
2. remove irrelevant sentences
3. preserve citations and headings
4. keep contradictory evidence separately
5. output provenance-tagged compressed snippets

### 7.4.4 Critical warning

Compression must never destroy the evidence required to justify the answer. A compression system must be evaluated against **support retention**, not token reduction alone.

---

## 7.5 Prompt Engineering for RAG

Prompt engineering in production RAG should be treated as **prompt compilation**, not handwritten template prose.

### 7.5.1 Prompt as runtime artifact

A high-quality prompt artifact contains:

1. system policy
2. task objective
3. answer contract
4. retrieved evidence
5. provenance/citation requirements
6. abstention behavior
7. tool constraints if iterative
8. output schema

Prompt construction objective:

$$
P^* = \arg\max_{P}
\Big[
\mathrm{InstructionClarity}(P)
+ \mathrm{EvidenceUtility}(P)
- \mathrm{TokenWaste}(P)
\Big]
$$

### 7.5.2 Evidence-grounded answer contract

The answer prompt should force:

1. use only cited evidence when making factual claims
2. identify uncertainty explicitly
3. abstain if evidence is insufficient
4. separate evidence from inference
5. preserve chronology and source conflicts

### 7.5.3 Chain of Thought

For RAG, Chain of Thought is most useful when:
1. evidence is conflicting
2. reasoning is multi-step
3. synthesis requires careful aggregation

Operationally, the preferred implementation is **bounded internal reasoning** or **structured verification checklists**, not unconstrained verbose reasoning. The target is not stylistic deliberation; it is evidence-grounded inference discipline.

### 7.5.4 Tree of Thoughts

Tree of Thoughts is appropriate when the system must evaluate multiple possible answer paths from diverse evidence subsets.

Example uses:
1. conflicting sources
2. diagnosis-style answers
3. strategic comparisons
4. multi-document synthesis

Tree search should be bounded by:
1. branch factor
2. depth
3. evaluation budget
4. latency class

### 7.5.5 ReAct

ReAct combines reasoning and action for iterative retrieval or tool use.

This is useful when the first retrieval pass is likely incomplete. The system can:
1. inspect retrieved evidence
2. identify missing facts
3. issue another retrieval/tool call
4. update the final answer

ReAct in RAG must be bounded by:
1. tool call budget
2. recursion depth
3. deadlines
4. verification gates

### 7.5.6 Recommended compiled prompt structure

```text
[ROLE POLICY]
- Answer only from provided evidence.
- Cite source identifiers for every factual claim.
- If evidence is insufficient, return INSUFFICIENT_EVIDENCE.

[TASK]
- Resolve user question with concise grounded synthesis.

[QUESTION]
{user_query}

[EVIDENCE]
- [doc_id:chunk_id|timestamp|title] snippet...
- [doc_id:chunk_id|timestamp|title] snippet...
- ...

[RESPONSE CONTRACT]
1. Direct answer
2. Supporting evidence with citations
3. Uncertainties / conflicts
4. If insufficient evidence, abstain
```

This design is substantially more robust than generic “use the context below” prompting.

---

## 7.6 LLM Fine-Tuning for RAG

LLM fine-tuning improves the generator when prompt engineering and retrieval optimization are insufficient for domain style, reasoning pattern, or answer formatting.

### 7.6.1 When LLM fine-tuning helps

1. domain-specific style or terminology
2. grounded answer formatting
3. citation discipline
4. structured synthesis
5. improved abstention behavior
6. tool-aware response behavior

### 7.6.2 When not to use it

Do not fine-tune the model to memorize fast-changing knowledge that should remain externalized in retrieval.

Knowledge should stay in:
1. indexes
2. metadata
3. tools
4. verified corpora

Fine-tuning should primarily improve:
1. generation policy
2. formatting
3. inference behavior
4. domain language handling

### 7.6.3 Retrieval-conditioned supervised fine-tuning

Train on tuples:
1. query
2. retrieved evidence
3. ideal grounded answer
4. citations
5. abstention cases

Supervised objective:

$$
\mathcal{L}_{\text{SFT}}
=
-\sum_{t=1}^{T}
\log p_{\theta}(y_t \mid q, E_q, y_{<t})
$$

### 7.6.4 Preference optimization

To improve groundedness and style, pair preferred grounded answers against hallucinated or unsupported alternatives.

A common objective is direct preference optimization:

$$
\mathcal{L}_{\text{DPO}}
=
-\log \sigma
\Big(
\beta
[
\log p_{\theta}(y^{+} \mid q, E_q)
-
\log p_{\theta}(y^{-} \mid q, E_q)
-
\log p_{\text{ref}}(y^{+} \mid q, E_q)
+
\log p_{\text{ref}}(y^{-} \mid q, E_q)
]
\Big)
$$

### 7.6.5 SOTA practice

1. retrieval-grounded SFT
2. preference tuning on faithfulness
3. synthetic data only with strict filtering
4. rejection sampling against citation and factuality validators
5. keep base model unchanged for volatile knowledge acquisition

---

## 7.7 Pseudo-Algorithm: Rerank, Enhance, Compress, Compile

```text
Algorithm 4: BUILD_EVIDENCE_PACK
Input:
  user_query q
  candidates C
  reranker R
  token_budget B
Output:
  evidence_pack E_final

C1 <- RERANK(R, q, C, top_n=N_rerank)

C2 <- DEDUP_NEAR_DUPLICATES(C1)
C3 <- APPLY_MMR(C2, lambda=lambda_mmr)

for each chunk c in C3 do
    c <- ENRICH_WITH_METADATA(c, fields=["title", "heading", "timestamp", "source"])
    if c.window_context exists then
        c <- EXPAND_TO_WINDOW(c)
    end if
end for

compressed <- []
remaining_budget <- B

for each chunk c in C3 do
    c_comp <- COMPRESS_CONTEXT(
        query=q,
        chunk=c,
        preserve_provenance=true,
        preserve_contradictions=true
    )
    if TOKENS(c_comp) <= remaining_budget then
        compressed.append(c_comp)
        remaining_budget <- remaining_budget - TOKENS(c_comp)
    end if
end for

E_final <- ORDER_BY_UTILITY_AND_DIVERSITY(compressed)
return E_final
```

---

# 8. Evaluation and Validation Pipeline

The report content concludes correctly only if the system includes measurable validation. Without stage-wise evaluation, optimization degenerates into anecdotal tuning.

## 8.1 Stage-Wise Metrics

### Indexing metrics

1. parsing success rate
2. structural preservation score
3. duplicate rate
4. chunk boundary quality
5. metadata completeness
6. ingestion idempotency correctness

### Retrieval metrics

1. Recall@k
2. MRR
3. nDCG@k
4. support coverage
5. lexical-exact-match hit rate
6. freshness-weighted relevance

Recall@k:

$$
\mathrm{Recall@k}
=
\frac{1}{N}
\sum_{i=1}^{N}
\mathbf{1}
[
G_i \cap \operatorname{topk}(q_i) \neq \emptyset
]
$$

where $$G_i$$ is the gold supporting set.

MRR:

$$
\mathrm{MRR}
=
\frac{1}{N}
\sum_{i=1}^{N}
\frac{1}{\mathrm{rank}_i}
$$

### Post-retrieval metrics

1. reranker lift over first-stage retrieval
2. context compression ratio
3. support retention after compression
4. citation coverage
5. contradiction preservation

### Generation metrics

1. answer correctness
2. faithfulness to evidence
3. citation precision
4. citation recall
5. abstention accuracy
6. hallucination rate

A faithfulness proxy can be defined as:

$$
\mathrm{Faithfulness}
=
\frac{1}{M}
\sum_{j=1}^{M}
\mathbf{1}
[
\text{claim}_j \text{ is supported by cited evidence}
]
$$

---

## 8.2 Validation Dataset Design

A serious RAG evaluation set must contain:

1. query
2. ideal answer
3. supporting passages
4. hard negatives
5. answerability label
6. temporal validity where relevant
7. metadata constraints
8. decomposition ground truth for complex queries

Include both:
1. single-hop tasks
2. multi-hop tasks

Also include adversarial cases:
1. semantically similar but wrong tenant
2. outdated versions
3. contradictory sources
4. exact-match identifiers
5. ambiguous entity names

---

## 8.3 Experimental Methodology

Every change should be evaluated by ablation, not by holistic anecdote.

Example ablation sequence:

1. baseline dense retrieval
2. + metadata filtering
3. + hybrid retrieval
4. + outlier exclusion
5. + reranking
6. + sentence window expansion
7. + compression
8. + prompt compiler improvement
9. + embedding fine-tuning
10. + LLM fine-tuning

Track:
1. metric delta
2. cost delta
3. latency delta
4. failure-class delta

---

# 9. Cross-Cutting Production Controls

The following controls are mandatory for enterprise-grade RAG, and each directly impacts the stages discussed above.

## 9.1 Hallucination Control

Hallucination is controlled primarily by retrieval discipline, not by prompt wording alone.

Required controls:

1. **provenance-tagged evidence only**
2. **citation-required answer contract**
3. **answerability classifier or insufficiency gate**
4. **conflict detection across sources**
5. **unsupported-claim suppression**
6. **post-generation verification against retrieved evidence**

A robust answer emission policy is:

- if support score is below threshold, abstain
- if sources conflict, state conflict explicitly
- if evidence is stale, flag temporal uncertainty

---

## 9.2 Fault Tolerance

Retrieval and indexing services must degrade predictably.

Required mechanisms:

1. retry budgets with jitter
2. circuit breakers on rerankers and LLMs
3. queue isolation for ingestion vs online retrieval
4. fallback from dense to lexical or hybrid-only modes
5. cached results for frequent queries
6. persisted failure traces for replay

---

## 9.3 Idempotency

Idempotency is critical for ingestion and stateful updates.

Use deterministic identifiers:

$$
\mathrm{chunk\_id} = H(\mathrm{document\_id}, \mathrm{offsets}, \mathrm{text}, \mathrm{schema\_version})
$$

This ensures that reprocessing the same document revision does not duplicate index entries.

---

## 9.4 Observability

Every stage must emit:

1. structured logs
2. latency metrics
3. retrieval traces
4. candidate counts
5. rerank deltas
6. token budgets
7. answerability scores
8. citation coverage
9. model/version identifiers

Minimum trace breakdown per request:

1. query normalization latency
2. rewrite/decompose latency
3. retrieval latency by index
4. fusion and filtering latency
5. reranker latency
6. compression latency
7. prompt token count
8. generation latency
9. final quality flags

---

## 9.5 Latency and Cost Optimization

Production RAG is a budgeted system.

### Latency controls

1. parallel subquery retrieval
2. cache query rewrites
3. ANN indexes for dense retrieval
4. rerank only over bounded candidate pools
5. compress before final LLM call
6. use small specialized models for rewriting and routing

### Token efficiency controls

1. sentence-window retrieval instead of large raw chunks
2. duplicate suppression
3. query-aware compression
4. deterministic prompt assembly
5. metadata selection only if useful

### Graceful degradation under load

When under load:

1. reduce expansion count
2. reduce rerank depth
3. switch to cheaper reranker
4. skip Tree-of-Thought branches
5. tighten token budgets
6. fallback to hybrid retrieval without iterative ReAct loops

Correct graceful degradation preserves correctness priorities while relaxing expensive refinements.

---

# 10. End-to-End SOTA Reference Pipeline

```text
Algorithm 5: HIGH_FIDELITY_RAG
Input:
  user_query q
  latency_budget B
  token_budget T
Output:
  grounded_answer a

// Pre-retrieval
plan <- PRE_RETRIEVAL_PLAN(q, user_context, B)

// Retrieval
candidates <- RETRIEVE_CANDIDATES(plan, dense_index, sparse_index, late_index)

// Post-retrieval
evidence_pack <- BUILD_EVIDENCE_PACK(q, candidates, reranker, T)

// Prompt compilation
prompt <- COMPILE_PROMPT(
    role_policy="grounded-answer-only",
    query=q,
    evidence=evidence_pack,
    answer_contract={
        citations_required: true,
        abstain_if_insufficient: true,
        conflict_reporting: true
    }
)

// Generation
draft <- GENERATE(prompt)

// Verification
verified <- VERIFY_CLAIMS_AGAINST_EVIDENCE(draft, evidence_pack)

if verified.support_score < TAU_SUPPORT then
    a <- BUILD_ABSTENTION_RESPONSE(evidence_pack)
else
    a <- ATTACH_CITATIONS_AND_EMIT(verified)
end if

return a
```

---

# 11. Recommended Technical Conclusions by Stage

## 11.1 Indexing

1. Treat parsing as structure recovery, not text extraction.
2. Preserve layout, hierarchy, and provenance.
3. Prefer document-aware or semantic chunking over fixed windows.
4. Use late chunking or proposition chunking for high-value corpora.
5. Attach metadata aggressively, but only if query-useful.

## 11.2 Pre-retrieval

1. Rewrite queries for retrievability while preserving intent.
2. Decompose multi-hop or multifactor questions into atomic subqueries.
3. Route by domain, modality, freshness, and answer type.
4. Bound expansion count by marginal recall gain.

## 11.3 Retrieval

1. Do not rely on dense top_k alone.
2. Use metadata filters for hard constraints.
3. Use hybrid retrieval as default, not exception.
4. Prune outliers using thresholding and autocut.
5. Fine-tune embeddings when corpus semantics diverge from general-purpose models.

## 11.4 Post-retrieval

1. Over-retrieve, then rerank.
2. Retrieve small units, generate from expanded windows.
3. Compress context without losing support evidence.
4. Compile prompts as deterministic runtime artifacts.
5. Fine-tune the LLM for grounded generation behavior, not volatile factual memory.

---

# 12. Final Architecture Position

The highest-performing RAG systems are not built by improving a single prompt or a single vector database call. They are built by engineering a deterministic evidence pipeline in which:

1. indexing preserves structure and provenance,
2. pre-retrieval transforms ambiguous intent into retrievable intent,
3. retrieval fuses semantic and lexical evidence under hard metadata constraints,
4. post-retrieval aggressively reranks, expands, and compresses context,
5. generation is governed by citation, abstention, and verification contracts.

That architecture is the operational path from naive RAG to production-grade grounded generation.



# ARAPO: Advanced Retrieval-Augmented Pipeline Orchestration — A Principal-Level Technical Report on State-of-the-Art RAG System Engineering

---

## Document Metadata

| Field | Value |
|---|---|
| **Document Class** | Principal-Level Technical Report |
| **Domain** | Retrieval-Augmented Generation, Agentic AI Systems, Context Engineering |
| **Audience** | Research Scientists, Principal Engineers, Agentic AI Architects |
| **Scope** | End-to-End SOTA RAG Pipeline: Indexing, Pre-Retrieval, Retrieval, Post-Retrieval |

---

## Table of Contents

1. [Formal Problem Statement & Pipeline Formalization](#1-formal-problem-statement--pipeline-formalization)
2. [Indexing Tier: Data Pre-Processing as a Typed Transformation Pipeline](#2-indexing-tier-data-pre-processing-as-a-typed-transformation-pipeline)
3. [Chunking Tier: Structure-Aware Segmentation with Provenance](#3-chunking-tier-structure-aware-segmentation-with-provenance)
4. [Pre-Retrieval Tier: Query Engineering as Compilation](#4-pre-retrieval-tier-query-engineering-as-compilation)
5. [Retrieval Tier: Deterministic Multi-Signal Retrieval Engine](#5-retrieval-tier-deterministic-multi-signal-retrieval-engine)
6. [Post-Retrieval Tier: Re-Ranking, Compression, and Generation Optimization](#6-post-retrieval-tier-re-ranking-compression-and-generation-optimization)
7. [Prompt Engineering as Compiled Runtime Artifact](#7-prompt-engineering-as-compiled-runtime-artifact)
8. [Generative Model Adaptation: Domain-Specific Fine-Tuning](#8-generative-model-adaptation-domain-specific-fine-tuning)
9. [Unified Pipeline Integration & Evaluation Infrastructure](#9-unified-pipeline-integration--evaluation-infrastructure)
10. [Conclusion & Architectural Invariants](#10-conclusion--architectural-invariants)

---

## 1. Formal Problem Statement & Pipeline Formalization

### 1.1 Definition

Retrieval-Augmented Generation (RAG) augments the parametric knowledge of a generative large language model $\mathcal{G}_\theta$ with non-parametric evidence retrieved from an external corpus $\mathcal{C} = \{d_1, d_2, \ldots, d_N\}$, reducing hallucination probability and anchoring generation to verifiable source material.

### 1.2 Formal Objective

Given a user query $q$, the RAG system seeks to produce a response $r^*$ that maximizes:

$$r^* = \arg\max_{r} \; P_{\mathcal{G}_\theta}\!\left(r \;\middle|\; q, \; \mathcal{R}(q, \mathcal{C})\right)$$

where $\mathcal{R}(q, \mathcal{C})$ is the retrieval function returning a ranked subset of evidence chunks from $\mathcal{C}$.

### 1.3 Naive RAG Decomposition

A naive pipeline decomposes into four operators:

$$\text{RAG}_{\text{naive}}(q) = \mathcal{G}_\theta\!\Big(\mathcal{T}\big(q, \; \text{TopK}\big(\text{sim}(\mathcal{E}(q), \; \mathcal{E}(\mathcal{C}))\big)\big)\Big)$$

where:
- $\mathcal{E}: \text{Text} \rightarrow \mathbb{R}^d$ is the embedding function
- $\text{sim}: \mathbb{R}^d \times \mathbb{R}^d \rightarrow \mathbb{R}$ is the similarity metric (typically cosine)
- $\text{TopK}$ selects the $k$ highest-scoring chunks
- $\mathcal{T}$ is the prompt template compiler
- $\mathcal{G}_\theta$ is the generative LLM

### 1.4 Failure Modes of Naive RAG

The naive pipeline suffers from the following formally characterizable failure modes:

| Failure Mode | Formal Description |
|---|---|
| **Semantic Gap** | $\text{sim}(\mathcal{E}(q), \mathcal{E}(d_{\text{relevant}})) < \tau$ despite $d_{\text{relevant}}$ being factually necessary |
| **Context Pollution** | $\exists \, d_j \in \text{TopK}$ such that $\text{relevance}(d_j, q) \approx 0$ but $\text{sim}(\mathcal{E}(q), \mathcal{E}(d_j)) > \tau$ |
| **Chunk Boundary Loss** | Semantically atomic information split across chunk boundaries, yielding incomplete evidence |
| **Query-Document Mismatch** | Colloquial or ambiguous $q$ maps poorly to formal document language in embedding space |
| **Freshness Decay** | Static index $\mathcal{C}$ diverges from ground truth over time $t$ |

### 1.5 Advanced RAG Pipeline — Typed Stage Decomposition

The advanced pipeline is formalized as a typed, multi-stage operator composition:

$$\text{RAG}_{\text{adv}}(q) = \mathcal{G}_{\theta'}\!\Big(\mathcal{P}\big(\mathcal{K}\big(\text{ReRank}\big(\mathcal{R}_{\text{hybrid}}\big(\mathcal{Q}(q)\big), q\big)\big)\big)\Big)$$

where each operator corresponds to a pipeline tier:

| Stage | Operator | Function |
|---|---|---|
| **Indexing** | $\mathcal{I}: \text{RawDocs} \rightarrow \mathcal{C}_{\text{indexed}}$ | Parse, clean, chunk, embed, store with metadata |
| **Pre-Retrieval** | $\mathcal{Q}: q \rightarrow \{q'_1, \ldots, q'_m\}$ | Transform, decompose, route queries |
| **Retrieval** | $\mathcal{R}_{\text{hybrid}}: \{q'_i\} \rightarrow \mathcal{D}_{\text{cand}}$ | Multi-signal retrieval with metadata filtering |
| **Post-Retrieval** | $\text{ReRank} \circ \mathcal{K} \circ \mathcal{P}$ | Re-rank, compress/enhance context, compile prompt |
| **Generation** | $\mathcal{G}_{\theta'}$ | Fine-tuned LLM generates grounded response |

---

## 2. Indexing Tier: Data Pre-Processing as a Typed Transformation Pipeline

### 2.1 Formal Data Flow

The indexing tier is a deterministic, DAG-structured transformation pipeline:

$$\text{RawSources} \xrightarrow{\text{Extract}} \text{ParsedDocs} \xrightarrow{\text{Clean}} \text{CleansedDocs} \xrightarrow{\text{Transform}} \text{StandardizedDocs} \xrightarrow{\text{Chunk}} \text{Chunks} \xrightarrow{\text{Embed+Index}} \mathcal{C}_{\text{indexed}}$$

Each stage is idempotent, provenance-tracked, and schema-versioned.

### 2.2 Data Extraction & Parsing — SOTA Multimodal Pipeline

#### 2.2.1 Extraction Protocol

Define the extraction operator for document $d$ of type $\tau$:

$$\text{Extract}(d, \tau) = \begin{cases} \text{StructuredParse}(d) & \tau \in \{\text{Markdown}, \text{HTML}, \text{JSON}\} \\ \text{LayoutLM-Parse}(d) & \tau \in \{\text{PDF}_{\text{digital}}\} \\ \text{ColPali/ColQwen}(d) & \tau \in \{\text{PDF}_{\text{scanned}}, \text{Image}\} \\ \text{DOM-Traverse}(d) & \tau = \text{WebHTML} \\ \text{TableRelation-Parse}(d) & \tau \in \{\text{XLSX}, \text{CSV}\} \end{cases}$$

#### 2.2.2 SOTA: Late-Interaction Multimodal Embedding for Document Understanding

ColPali and ColQwen represent a paradigm shift from OCR-then-embed to direct visual embedding:

$$\mathbf{V}_{\text{page}} = \text{ColPali}(\text{PageImage}) \in \mathbb{R}^{n_{\text{patches}} \times d}$$

$$\text{Score}(q, \text{page}) = \sum_{i=1}^{|q|} \max_{j=1}^{n_{\text{patches}}} \; \mathbf{q}_i^\top \mathbf{V}_{\text{page}, j}$$

This MaxSim late-interaction scoring eliminates OCR error propagation entirely, preserving spatial layout information (tables, figures, headers) natively.

#### 2.2.3 Metadata Extraction Contract

Every extracted element emits a typed metadata record:

```
MetadataRecord := {
    source_uri:     URI,
    doc_type:       Enum[PDF|HTML|MD|XLSX|...],
    author:         Optional[String],
    timestamp:      ISO8601,
    section_path:   List[String],      // hierarchical section trace
    page_number:    Optional[Int],
    extraction_model: String,          // provenance of extraction method
    confidence:     Float[0,1],
    content_hash:   SHA256
}
```

### 2.3 Data Cleaning — Information-Theoretic Noise Reduction

#### 2.3.1 Formal Objective

Minimize noise entropy while preserving information content:

$$\text{Clean}(d) = \arg\min_{d'} \; H_{\text{noise}}(d') \quad \text{s.t.} \quad \text{MI}(d'; d_{\text{signal}}) \geq (1 - \epsilon) \cdot \text{MI}(d; d_{\text{signal}})$$

where $H_{\text{noise}}$ is the entropy attributable to boilerplate, formatting artifacts, and structural noise, $\text{MI}$ denotes mutual information with the ground-truth signal content, and $\epsilon$ is a configurable loss tolerance.

#### 2.3.2 SOTA Cleaning Operations

| Operation | Technique | Detail |
|---|---|---|
| Boilerplate Removal | Content-to-boilerplate classifier (fine-tuned DistilBERT) | Per-paragraph classification with $P(\text{boilerplate}) > 0.9$ threshold |
| Deduplication | MinHash LSH with Jaccard threshold $J > 0.85$ | Exact and near-duplicate removal across corpus |
| Encoding Normalization | Unicode NFKC + whitespace canonicalization | Deterministic text normalization |
| Table Linearization | Structure-aware serialization preserving row-column relationships | Markdown table format with column headers repeated per row for retrieval fidelity |
| Language Detection | FastText LID with confidence gating | Route non-target-language content to specialized pipelines or filter |

### 2.4 Data Transformation — Schema Standardization

Transform all cleaned content into a canonical document model:

```
StandardizedDocument := {
    doc_id:        UUID,
    content:       String,
    elements:      List[Element],       // paragraphs, tables, code blocks, lists
    metadata:      MetadataRecord,
    lineage:       TransformationTrace  // full provenance chain
}

Element := {
    type:          Enum[PARAGRAPH|TABLE|CODE|LIST|HEADING],
    content:       String,
    position:      Int,                 // ordinal within document
    parent_section: Optional[String]
}
```

### 2.5 Pseudo-Algorithm: End-to-End Indexing Pipeline

```
Algorithm 1: ARAPO-IndexingPipeline
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:  RawSources S = {s₁, s₂, ..., sₙ}
Output: Indexed corpus C_indexed with provenance

1:  for each source sᵢ ∈ S do
2:      τ ← DetectDocumentType(sᵢ)
3:      d_parsed ← Extract(sᵢ, τ)                    // type-dispatched extraction
4:      meta ← ExtractMetadata(sᵢ, d_parsed, τ)
5:      d_clean ← ApplyCleaningPipeline(d_parsed)     // boilerplate, dedup, normalize
6:      if DeduplicateCheck(d_clean, C_existing) then
7:          SKIP sᵢ; LOG("near-duplicate", hash(d_clean))
8:          continue
9:      end if
10:     d_std ← TransformToSchema(d_clean, meta)
11:     chunks ← ChunkingStrategy(d_std, strategy=SelectOptimal(τ, |d_std|))
12:     for each chunk cⱼ ∈ chunks do
13:         cⱼ.embedding ← EmbeddingModel(cⱼ.content)
14:         cⱼ.metadata ← EnrichMetadata(meta, cⱼ.position, cⱼ.section_path)
15:         cⱼ.provenance ← BuildLineageTrace(sᵢ, pipeline_version)
16:         VectorStore.Upsert(cⱼ.id, cⱼ.embedding, cⱼ.metadata, cⱼ.content)
17:     end for
18: end for
19: return C_indexed
```

---

## 3. Chunking Tier: Structure-Aware Segmentation with Provenance

### 3.1 Formal Chunking Problem

Given a standardized document $D = [e_1, e_2, \ldots, e_L]$ of $L$ elements, partition $D$ into chunks $\{c_1, c_2, \ldots, c_K\}$ that optimize the retrieval utility function:

$$\{c_1^*, \ldots, c_K^*\} = \arg\max_{\{c_k\}} \sum_{k=1}^{K} \Big[\underbrace{\text{SemCoherence}(c_k)}_{\text{intra-chunk semantic unity}} + \lambda_1 \cdot \underbrace{\text{RetPrec}(c_k, \mathcal{Q})}_{\text{retrieval precision}} - \lambda_2 \cdot \underbrace{\text{InfoLoss}(c_k, D)}_{\text{boundary information loss}}\Big]$$

subject to:

$$\forall k: \; |c_k|_{\text{tokens}} \leq T_{\max} \quad \text{and} \quad \bigcup_{k} c_k = D$$

where $T_{\max}$ is the maximum token budget per chunk.

### 3.2 SOTA Chunking Strategy Taxonomy

#### 3.2.1 Fixed-Size Chunking with Overlap

Partition $D$ into chunks of size $s$ tokens with overlap $o$:

$$c_i = D[i \cdot (s - o) : i \cdot (s - o) + s], \quad i = 0, 1, 2, \ldots$$

**Overlap ratio:**

$$\rho = \frac{o}{s}, \quad \text{typically } \rho \in [0.1, 0.25]$$

**Limitation:** Agnostic to semantic boundaries. Suitable only as a baseline or for highly homogeneous corpora.

#### 3.2.2 Recursive Hierarchical Chunking

Apply a priority-ordered separator hierarchy $\Sigma = [\sigma_1, \sigma_2, \ldots, \sigma_m]$ (e.g., $[\text{section}, \text{paragraph}, \text{sentence}, \text{word}]$):

```
Algorithm 2: RecursiveChunk
━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:  text T, separators Σ = [σ₁, ..., σₘ], max_size T_max
Output: chunks C

1:  if |T|_tokens ≤ T_max then
2:      return {T}
3:  end if
4:  for i = 1 to m do
5:      segments ← Split(T, σᵢ)
6:      if max(|seg|_tokens for seg ∈ segments) ≤ T_max then
7:          C ← MergeAdjacentUnderBudget(segments, T_max)
8:          return C
9:      else
10:         C ← ∅
11:         for each seg ∈ segments do
12:             C ← C ∪ RecursiveChunk(seg, Σ[i+1:], T_max)
13:         end for
14:         return C
15:     end if
16: end for
17: return FixedSizeChunk(T, T_max)   // fallback
```

#### 3.2.3 Semantic Chunking via Embedding Breakpoint Detection

This SOTA technique identifies semantic shift boundaries by computing inter-sentence embedding distances:

**Step 1:** Embed each sentence $s_i$ independently:

$$\mathbf{e}_i = \mathcal{E}(s_i) \in \mathbb{R}^d$$

**Step 2:** Compute pairwise cosine distance between consecutive sentences:

$$\delta_i = 1 - \frac{\mathbf{e}_i \cdot \mathbf{e}_{i+1}}{\|\mathbf{e}_i\| \cdot \|\mathbf{e}_{i+1}\|}$$

**Step 3:** Detect breakpoints where $\delta_i$ exceeds a threshold derived from the distribution:

$$\text{Breakpoint at } i \quad \text{if} \quad \delta_i > \mu_\delta + \alpha \cdot \sigma_\delta$$

where $\mu_\delta$ and $\sigma_\delta$ are the mean and standard deviation of the distance series, and $\alpha$ is the sensitivity hyperparameter (typically $\alpha \in [0.5, 2.0]$).

```
Algorithm 3: SemanticChunking
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:  document D, embedding model E, sensitivity α, max_size T_max
Output: semantically coherent chunks C

1:  sentences ← SentenceTokenize(D)
2:  embeddings ← [E(sᵢ) for sᵢ ∈ sentences]
3:  distances ← [CosineDist(embeddings[i], embeddings[i+1])
                   for i = 0 to |sentences|-2]
4:  μ ← Mean(distances)
5:  σ ← StdDev(distances)
6:  threshold ← μ + α · σ
7:  breakpoints ← {i : distances[i] > threshold}
8:  chunks ← PartitionAtBreakpoints(sentences, breakpoints)
9:  // Post-hoc size enforcement
10: C ← ∅
11: for each chunk c ∈ chunks do
12:     if |c|_tokens > T_max then
13:         C ← C ∪ RecursiveChunk(c, Σ_default, T_max)
14:     else
15:         C ← C ∪ {c}
16:     end if
17: end for
18: return C
```

#### 3.2.4 Late Chunking — SOTA Context-Preserving Method

Late chunking reverses the traditional embed-after-chunk pipeline by processing the full document through a long-context embedding model first, then performing chunking on the contextualized token embeddings:

**Step 1:** Pass entire document through a long-context encoder:

$$\mathbf{H} = \text{LongContextEncoder}(D) \in \mathbb{R}^{L \times d}$$

where $L$ is the total token length and each $\mathbf{h}_i$ carries attention from the full document context.

**Step 2:** Apply structural or semantic boundaries to partition $\mathbf{H}$ into chunk spans:

$$c_k = \mathbf{H}[b_k : b_{k+1}], \quad b_k \in \text{BoundarySet}$$

**Step 3:** Pool each chunk span to produce the chunk embedding:

$$\mathbf{e}_{c_k} = \frac{1}{b_{k+1} - b_k} \sum_{i=b_k}^{b_{k+1}-1} \mathbf{h}_i$$

**Advantage:** Each chunk embedding $\mathbf{e}_{c_k}$ retains cross-document attention context, eliminating the information loss inherent in embed-after-chunk pipelines.

```
Algorithm 4: LateChunking
━━━━━━━━━━━━━━━━━━━━━━━━━
Input:  document D, long-context encoder M, boundary_strategy B
Output: chunks with context-aware embeddings

1:  tokens ← Tokenize(D)
2:  ASSERT |tokens| ≤ M.max_context_length
3:  H ← M.Encode(tokens)                    // H ∈ ℝ^{L×d}, full-document attention
4:  boundaries ← B.DetectBoundaries(D)       // structural or semantic
5:  C ← ∅
6:  for k = 0 to |boundaries|-1 do
7:      span ← H[boundaries[k] : boundaries[k+1]]
8:      embedding ← MeanPool(span)
9:      chunk_text ← Detokenize(tokens[boundaries[k]:boundaries[k+1]])
10:     C ← C ∪ {(chunk_text, embedding, provenance=(D.id, k, boundaries))}
11: end for
12: return C
```

#### 3.2.5 Agentic Chunking — LLM-Driven Proposition Extraction

An LLM decomposes the document into atomic, self-contained propositions:

$$D \xrightarrow{\mathcal{G}_\theta} \{p_1, p_2, \ldots, p_M\}$$

where each $p_i$ is a factual statement that is independently interpretable without requiring surrounding context.

Propositions are then clustered into chunks via semantic similarity:

$$\text{Cluster}(\{p_i\}) \rightarrow \{c_1, \ldots, c_K\} \quad \text{via agglomerative clustering on } \mathcal{E}(p_i)$$

**Trade-off:** Highest retrieval precision but $O(N \cdot T_{\text{LLM}})$ computational cost, where $T_{\text{LLM}}$ is per-token inference cost. Reserved for high-value, slowly-changing corpora.

### 3.3 Chunking Strategy Selection Matrix

| Document Class | Recommended Strategy | Rationale |
|---|---|---|
| Homogeneous prose (articles, blogs) | Semantic chunking ($\alpha=1.0$) | Preserves topic boundaries |
| Structured documents (HTML, Markdown, code) | Document-based / Recursive hierarchical | Exploits explicit structural markers |
| Long technical documents (research papers) | Late chunking | Requires cross-section context preservation |
| Legal/medical regulatory text | Agentic (proposition extraction) | Demands atomic, self-contained facts |
| Mixed-format enterprise corpora | Hybrid: type-dispatch to above strategies | Per-document-class routing |

---

## 4. Pre-Retrieval Tier: Query Engineering as Compilation

### 4.1 Formal Objective

Transform a raw user query $q$ into an optimized query set $\mathcal{Q}^* = \{q'_1, \ldots, q'_m\}$ that maximizes retrieval recall and precision:

$$\mathcal{Q}^*(q) = \arg\max_{\{q'_i\}} \sum_{i=1}^{m} \text{RetrievalUtility}\big(\mathcal{R}(q'_i, \mathcal{C}), \; q\big)$$

subject to latency and token budget constraints.

### 4.2 Query Rewriting — Rewrite-Retrieve-Read Paradigm

#### 4.2.1 Formalization

Define the rewriting operator:

$$q' = \text{Rewrite}_\phi(q, \; \mathcal{M}_{\text{session}})$$

where $\phi$ parameterizes the rewriter (LLM or specialized small model) and $\mathcal{M}_{\text{session}}$ is session memory providing conversational context for coreference resolution and intent clarification.

The objective is to minimize the semantic gap:

$$\text{Rewrite}_\phi^* = \arg\min_\phi \; \mathbb{E}_{q \sim \mathcal{Q}_{\text{user}}}\!\Big[\text{dist}\!\big(\mathcal{E}(q'), \; \mathcal{E}(d_{\text{gold}})\big)\Big]$$

where $d_{\text{gold}}$ is the ground-truth relevant document.

#### 4.2.2 SOTA Implementation: Trainable Query Rewriter

Rather than relying on generic LLM prompting, deploy a specialized rewriter model fine-tuned on $(q_{\text{raw}}, q_{\text{optimal}})$ pairs derived from retrieval logs:

$$\mathcal{L}_{\text{rewriter}} = -\sum_{(q, q^*)} \log P_\phi(q^* | q)$$

where $q^*$ is the query formulation that historically yielded the highest retrieval precision.

```
Algorithm 5: QueryRewrite
━━━━━━━━━━━━━━━━━━━━━━━━
Input:  raw query q, session memory M_session, rewriter model φ
Output: optimized query q'

1:  context ← ExtractRelevantHistory(M_session, q, window=3)
2:  q_resolved ← CoreferenceResolve(q, context)
3:  q' ← φ.Generate(
         instruction="Reformulate for maximum retrieval precision against a "
                     "technical knowledge base. Preserve intent. Use precise "
                     "domain terminology.",
         input=q_resolved
     )
4:  VALIDATE: AssertSemanticPreservation(q, q')  // cosine(E(q), E(q')) > 0.7
5:  return q'
```

### 4.3 Query Expansion — Multi-Perspective Retrieval

#### 4.3.1 Formalization

Generate $m$ diverse query variants to expand recall:

$$\{q'_1, \ldots, q'_m\} = \text{Expand}_\phi(q), \quad m \in [3, 6]$$

Each variant targets a distinct facet of the information need. The combined retrieval set is:

$$\mathcal{D}_{\text{expanded}} = \bigcup_{i=1}^{m} \text{TopK}_i\!\big(\mathcal{R}(q'_i, \mathcal{C})\big)$$

#### 4.3.2 Diversity Enforcement

To avoid redundant variants, enforce minimum pairwise diversity:

$$\forall \, i \neq j: \quad 1 - \cos\!\big(\mathcal{E}(q'_i), \mathcal{E}(q'_j)\big) > \delta_{\text{div}}, \quad \delta_{\text{div}} \in [0.2, 0.4]$$

```
Algorithm 6: QueryExpansion
━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:  query q, expansion_count m, diversity_threshold δ_div
Output: expanded query set Q' = {q'₁, ..., q'ₘ}

1:  candidates ← LLM.Generate(
        instruction="Generate m diverse reformulations targeting different "
                    "aspects of the information need.",
        input=q, n=2*m   // over-generate for diversity selection
    )
2:  Q' ← {candidates[0]}
3:  for each c ∈ candidates[1:] do
4:      if ∀ q' ∈ Q': CosineDist(E(c), E(q')) > δ_div then
5:          Q' ← Q' ∪ {c}
6:      end if
7:      if |Q'| ≥ m then break
8:  end for
9:  return Q'
```

### 4.4 Query Decomposition — Divide-and-Conquer Retrieval

#### 4.4.1 Formal Decomposition

For a complex query $q_{\text{complex}}$, decompose into independent sub-queries:

$$\text{Decompose}(q_{\text{complex}}) = \{q_{\text{sub},1}, \ldots, q_{\text{sub},n}\}$$

such that:

$$\text{Answer}(q_{\text{complex}}) = \text{Synthesize}\!\Big(\bigcup_{i=1}^{n} \text{Answer}(q_{\text{sub},i})\Big)$$

#### 4.4.2 Decomposition with Dependency Graph

SOTA decomposition produces a dependency DAG rather than a flat list:

$$G_{\text{decomp}} = (V, E), \quad V = \{q_{\text{sub},i}\}, \quad E = \{(q_i, q_j) \mid q_j \text{ depends on answer to } q_i\}$$

Independent sub-queries execute in parallel; dependent sub-queries execute after their predecessors resolve.

```
Algorithm 7: QueryDecomposition
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:  complex query q, LLM decomposer D
Output: sub-query DAG G = (V, E), final synthesized answer

STAGE I: DECOMPOSITION
1:  decomposition ← D.Generate(
        instruction="Decompose into independent or sequentially dependent "
                    "sub-queries. Output JSON: [{id, query, depends_on: [ids]}]",
        input=q
    )
2:  V ← {sq.query for sq ∈ decomposition}
3:  E ← {(sq_dep, sq) for sq ∈ decomposition for sq_dep ∈ sq.depends_on}
4:  G ← DAG(V, E)

STAGE II: PARALLEL RETRIEVAL + RESOLUTION
5:  topological_order ← TopologicalSort(G)
6:  results ← {}
7:  for each level ∈ topological_order do
8:      PARALLEL for each sq ∈ level do
9:          context_sq ← Retrieve(sq.query, C)
10:         prior_answers ← {results[dep] for dep ∈ sq.depends_on}
11:         results[sq.id] ← LLM.Answer(sq.query, context_sq, prior_answers)
12:     end PARALLEL
13: end for

STAGE III: SYNTHESIS
14: final_answer ← LLM.Synthesize(q, results)
15: return final_answer
```

### 4.5 Query Routing — Intent-Aware Pipeline Selection

#### 4.5.1 Formal Routing Function

Define a routing classifier $\mathcal{R}_{\text{route}}$:

$$\mathcal{R}_{\text{route}}(q) \rightarrow \pi_j \in \Pi = \{\pi_1, \ldots, \pi_P\}$$

where $\Pi$ is the set of available retrieval pipelines, each optimized for a specific query class.

#### 4.5.2 Multi-Index Routing Architecture

| Pipeline $\pi_j$ | Query Class | Retrieval Strategy |
|---|---|---|
| $\pi_{\text{factual}}$ | Factual lookup | Exact-match + dense retrieval on factual index |
| $\pi_{\text{analytical}}$ | Analytical/comparison | Multi-hop retrieval with sub-query decomposition |
| $\pi_{\text{temporal}}$ | Time-sensitive | Freshness-weighted retrieval with timestamp filters |
| $\pi_{\text{structured}}$ | Structured data queries | Text-to-SQL / Text-to-API conversion |
| $\pi_{\text{web}}$ | Out-of-corpus queries | Live web search with result verification |
| $\pi_{\text{code}}$ | Code-related queries | AST-aware code search + documentation retrieval |

#### 4.5.3 Agentic Routing Protocol

```
Algorithm 8: AgenticQueryRouter
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:  query q, pipeline registry Π, routing model R_route
Output: pipeline selection π*, routed query q'

1:  features ← ExtractQueryFeatures(q)     // intent, complexity, domain, entities
2:  π_scores ← R_route.Classify(features)  // probability distribution over Π
3:  π* ← argmax(π_scores)
4:  if max(π_scores) < confidence_threshold then
5:      // Low-confidence: engage meta-agent for deliberation
6:      π* ← MetaAgent.Deliberate(q, Π, π_scores)
7:  end if
8:  q' ← π*.QueryAdapter(q)               // pipeline-specific query transformation
9:  LOG(trace_id, q, π*, π_scores)         // observability
10: return (π*, q')
```

---

## 5. Retrieval Tier: Deterministic Multi-Signal Retrieval Engine

### 5.1 Metadata Filtering — Structured Pre-Filtering

#### 5.1.1 Formal Filter Specification

Define a metadata filter as a predicate function:

$$f_{\text{meta}}: \text{MetadataRecord} \rightarrow \{0, 1\}$$

The filtered candidate set before vector search:

$$\mathcal{C}_{\text{filtered}} = \{c \in \mathcal{C} \mid f_{\text{meta}}(c.\text{metadata}) = 1\}$$

Vector search then operates exclusively on $\mathcal{C}_{\text{filtered}}$:

$$\text{Results} = \text{TopK}\!\big(\text{sim}(\mathcal{E}(q), \mathcal{E}(c)), \; c \in \mathcal{C}_{\text{filtered}}\big)$$

#### 5.1.2 Filter Composition Algebra

Filters compose via boolean algebra:

$$f_{\text{composite}} = f_1 \land f_2 \lor \lnot f_3$$

**Example:** For a query from user Alice about recent Python documentation:

$$f = (\text{user} = \text{"Alice"}) \land (\text{language} = \text{"Python"}) \land (\text{timestamp} \geq t_{\text{now}} - 90d)$$

#### 5.1.3 Automated Filter Extraction

SOTA systems extract metadata filters directly from the query using an LLM:

```
Algorithm 9: AutoMetadataFilterExtraction
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:  query q, metadata schema S
Output: filter predicate f_meta

1:  extraction ← LLM.Generate(
        instruction="Given the query and metadata schema, extract applicable "
                    "filter predicates as structured JSON. Schema: " + S,
        input=q
    )
2:  f_meta ← ParseAndValidateFilter(extraction, S)
3:  if f_meta.EstimatedSelectivity(C) < min_candidates then
4:      f_meta ← RelaxLeastConstraint(f_meta)  // prevent empty result sets
5:  end if
6:  return f_meta
```

### 5.2 Vector Search Outlier Exclusion

#### 5.2.1 Distance Thresholding

Given a distance function $d(q, c)$ and threshold $\tau_d$:

$$\mathcal{D}_{\text{valid}} = \{c \in \text{TopK}(q) \mid d(q, c) \leq \tau_d\}$$

For cosine similarity $s \in [-1, 1]$, the equivalent condition:

$$\mathcal{D}_{\text{valid}} = \{c \mid \cos(\mathcal{E}(q), \mathcal{E}(c)) \geq \tau_s\}$$

**Threshold calibration:** Compute $\tau_s$ from a validation set of known-relevant queries:

$$\tau_s = \text{Percentile}_{p}\!\big(\{\cos(\mathcal{E}(q_i), \mathcal{E}(d_i^{\text{gold}}))\}_{i=1}^{N_{\text{val}}}\big), \quad p \in [5, 15]$$

#### 5.2.2 Autocut — Adaptive Cluster-Based Cutoff

Detect natural discontinuities in the sorted distance sequence:

$$\Delta_i = d_{(i+1)} - d_{(i)}, \quad i = 1, \ldots, K-1$$

where $d_{(i)}$ is the $i$-th sorted distance. Cut at the first significant gap:

$$i^* = \min\{i \mid \Delta_i > \mu_\Delta + \beta \cdot \sigma_\Delta\}$$

$$\mathcal{D}_{\text{autocut}} = \{c_{(1)}, \ldots, c_{(i^*)}\}$$

```
Algorithm 10: Autocut
━━━━━━━━━━━━━━━━━━━━
Input:  sorted results R = [(c₁,d₁), ..., (cₖ,dₖ)], sensitivity β
Output: filtered results R_cut

1:  distances ← [d₁, d₂, ..., dₖ]  // already sorted ascending
2:  gaps ← [distances[i+1] - distances[i] for i = 0 to K-2]
3:  μ_gap ← Mean(gaps)
4:  σ_gap ← StdDev(gaps)
5:  cutoff_threshold ← μ_gap + β · σ_gap
6:  cut_index ← K   // default: keep all
7:  for i = 0 to |gaps|-1 do
8:      if gaps[i] > cutoff_threshold then
9:          cut_index ← i + 1
10:         break
11:     end if
12: end for
13: R_cut ← R[0 : cut_index]
14: return R_cut
```

### 5.3 Hybrid Search — Multi-Signal Fusion

#### 5.3.1 Formal Hybrid Retrieval

Combine dense (semantic) and sparse (lexical) retrieval via a convex combination:

$$\text{Score}_{\text{hybrid}}(q, c) = \alpha \cdot \text{Score}_{\text{semantic}}(q, c) + (1 - \alpha) \cdot \text{Score}_{\text{lexical}}(q, c)$$

where:

- $\text{Score}_{\text{semantic}}(q, c) = \cos\!\big(\mathcal{E}_{\text{dense}}(q), \mathcal{E}_{\text{dense}}(c)\big)$
- $\text{Score}_{\text{lexical}}(q, c) = \text{BM25}(q, c)$
- $\alpha \in [0, 1]$ is the fusion weight

#### 5.3.2 BM25 Scoring Function

$$\text{BM25}(q, c) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t, c) \cdot (k_1 + 1)}{f(t, c) + k_1 \cdot \left(1 - b + b \cdot \frac{|c|}{|\overline{c}|}\right)}$$

where $f(t, c)$ is the term frequency of $t$ in $c$, $|c|$ is the chunk length, $|\overline{c}|$ is the average chunk length, and $k_1$, $b$ are hyperparameters (typically $k_1 = 1.2$, $b = 0.75$).

The inverse document frequency:

$$\text{IDF}(t) = \log\!\left(\frac{N - n(t) + 0.5}{n(t) + 0.5} + 1\right)$$

where $N$ is the total number of chunks and $n(t)$ is the number of chunks containing term $t$.

#### 5.3.3 SOTA Fusion: Reciprocal Rank Fusion (RRF)

Rather than score-level interpolation, RRF operates on rank positions, which is more robust to score distribution mismatch:

$$\text{RRF}(c) = \sum_{r \in \mathcal{R}_{\text{lists}}} \frac{1}{\kappa + \text{rank}_r(c)}$$

where $\kappa$ is a smoothing constant (typically $\kappa = 60$) and $\text{rank}_r(c)$ is the rank of chunk $c$ in retrieval list $r$.

```
Algorithm 11: HybridSearchWithRRF
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:  query q, vector index V, keyword index K, κ=60, top_k
Output: fused ranked results

1:  R_semantic ← V.Search(E_dense(q), top_k=3*top_k)
2:  R_lexical  ← K.Search(q, top_k=3*top_k)           // BM25
3:  
4:  // Reciprocal Rank Fusion
5:  scores ← DefaultDict(0.0)
6:  for each (rank, chunk) ∈ Enumerate(R_semantic) do
7:      scores[chunk.id] += 1.0 / (κ + rank + 1)
8:  end for
9:  for each (rank, chunk) ∈ Enumerate(R_lexical) do
10:     scores[chunk.id] += 1.0 / (κ + rank + 1)
11: end for
12: fused ← SortByScoreDescending(scores)
13: return fused[:top_k]
```

#### 5.3.4 Alpha Tuning Protocol

Optimal $\alpha$ depends on query-type distribution and corpus characteristics:

| Corpus/Query Characteristic | Recommended $\alpha$ | Justification |
|---|---|---|
| Technical terminology-heavy (code, medical) | $0.3$–$0.5$ | Keyword precision critical for domain terms |
| Conversational / paraphrased queries | $0.7$–$0.9$ | Semantic understanding dominates |
| Mixed enterprise corpus | $0.5$–$0.6$ | Balanced default |

Use RRF when score distributions are incomparable across retrieval signals.

### 5.4 Embedding Model Fine-Tuning

#### 5.4.1 Problem Formulation

Given a pre-trained embedding model $\mathcal{E}_{\theta_0}$ and a domain-specific dataset $\mathcal{D}_{\text{domain}} = \{(q_i, d_i^+, d_i^-)\}_{i=1}^{N}$ of query-positive-negative triplets, fine-tune to obtain $\mathcal{E}_{\theta^*}$:

$$\theta^* = \arg\min_\theta \; \mathbb{E}_{(q, d^+, d^-) \sim \mathcal{D}_{\text{domain}}} \Big[\mathcal{L}_{\text{contrastive}}\!\big(\mathcal{E}_\theta(q), \mathcal{E}_\theta(d^+), \mathcal{E}_\theta(d^-)\big)\Big]$$

#### 5.4.2 SOTA Loss Functions

**InfoNCE Loss (Multi-Negative):**

$$\mathcal{L}_{\text{InfoNCE}} = -\log \frac{\exp\!\big(\text{sim}(\mathbf{q}, \mathbf{d}^+) / \tau\big)}{\exp\!\big(\text{sim}(\mathbf{q}, \mathbf{d}^+) / \tau\big) + \sum_{j=1}^{N_{\text{neg}}} \exp\!\big(\text{sim}(\mathbf{q}, \mathbf{d}_j^-) / \tau\big)}$$

where $\tau$ is the temperature parameter controlling distribution sharpness.

**MNRL (Multiple Negatives Ranking Loss):**

$$\mathcal{L}_{\text{MNRL}} = -\frac{1}{B} \sum_{i=1}^{B} \log \frac{\exp\!\big(\text{sim}(\mathbf{q}_i, \mathbf{d}_i^+) / \tau\big)}{\sum_{j=1}^{B} \exp\!\big(\text{sim}(\mathbf{q}_i, \mathbf{d}_j^+) / \tau\big)}$$

where the in-batch positives for other queries serve as negatives (in-batch negative mining).

**Hard Negative Mining:**

Select negatives that are semantically close but factually incorrect:

$$d^-_{\text{hard}} = \arg\max_{d \in \mathcal{C} \setminus \{d^+\}} \; \text{sim}\!\big(\mathcal{E}_{\theta_0}(q), \mathcal{E}_{\theta_0}(d)\big)$$

Hard negatives force the model to learn fine-grained discriminations rather than trivially separating dissimilar documents.

#### 5.4.3 Fine-Tuning Pipeline

```
Algorithm 12: EmbeddingModelFineTuning
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:  base model E_θ₀, domain dataset D, validation set V, epochs E
Output: fine-tuned model E_θ*

1:  // Training data construction
2:  triplets ← ∅
3:  for each (qᵢ, dᵢ⁺) ∈ D do
4:      negatives ← HardNegativeMine(qᵢ, C, E_θ₀, n_hard=5)
5:      in_batch_neg ← True   // enable in-batch negative mining
6:      triplets ← triplets ∪ {(qᵢ, dᵢ⁺, negatives)}
7:  end for
8:  
9:  // Fine-tuning loop
10: θ ← θ₀
11: for epoch = 1 to E do
12:     for each batch B ∈ Batches(triplets) do
13:         L ← ComputeInfoNCELoss(B, E_θ, τ=0.05)
14:         θ ← θ - η · ∇_θ L
15:     end for
16:     // Validation
17:     metrics ← EvaluateRetrieval(E_θ, V)   // MRR, Recall@k, NDCG
18:     if metrics.MRR > best_MRR then
19:         SaveCheckpoint(E_θ)
20:         best_MRR ← metrics.MRR
21:     end if
22:     if EarlyStopping(metrics) then break
23: end for
24: return E_θ*
```

#### 5.4.4 Evaluation Metrics for Fine-Tuned Embeddings

**Mean Reciprocal Rank (MRR):**

$$\text{MRR} = \frac{1}{|\mathcal{Q}_{\text{val}}|} \sum_{i=1}^{|\mathcal{Q}_{\text{val}}|} \frac{1}{\text{rank}(d_i^+)}$$

**Recall@K:**

$$\text{Recall@K} = \frac{1}{|\mathcal{Q}_{\text{val}}|} \sum_{i=1}^{|\mathcal{Q}_{\text{val}}|} \mathbb{1}\!\big[d_i^+ \in \text{TopK}(\mathcal{R}(q_i))\big]$$

**Normalized Discounted Cumulative Gain (NDCG@K):**

$$\text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}, \quad \text{DCG@K} = \sum_{i=1}^{K} \frac{2^{\text{rel}_i} - 1}{\log_2(i + 1)}$$

---

## 6. Post-Retrieval Tier: Re-Ranking, Compression, and Generation Optimization

### 6.1 Re-Ranking — Cross-Encoder Precision Refinement

#### 6.1.1 Bi-Encoder vs. Cross-Encoder Trade-off

**Bi-encoder (retrieval stage):** encodes $q$ and $c$ independently; $O(1)$ per query after pre-indexing:

$$\text{score}_{\text{bi}} = \cos\!\big(\mathcal{E}(q), \mathcal{E}(c)\big)$$

**Cross-encoder (re-ranking stage):** processes $(q, c)$ jointly through full attention:

$$\text{score}_{\text{cross}} = \sigma\!\big(W^\top \cdot \text{CLS}(\text{Encoder}([q; \text{SEP}; c]))\big)$$

where $\sigma$ is the sigmoid function and $\text{CLS}$ is the classification token representation.

**Computational asymmetry:**

| Model | Inference Cost | Contextual Fidelity |
|---|---|---|
| Bi-encoder | $O(d)$ dot product | Low (independent encoding) |
| Cross-encoder | $O(|q| \cdot |c| \cdot L)$ transformer attention | High (joint attention) |

#### 6.1.2 Retrieve-and-Rerank Pipeline

$$\text{TopK}_{\text{final}} = \text{ReRank}\!\Big(\text{TopN}_{\text{retrieve}}(q, \mathcal{C}), \; q, \; k\Big), \quad N \gg k$$

The over-retrieval ratio $\gamma = N / k$ (typically $\gamma \in [3, 10]$) determines the recall-precision trade-off.

```
Algorithm 13: RetrieveAndRerank
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:  query q, vector store V, cross-encoder CE,
        retrieve_n N, final_k k
Output: re-ranked top-k chunks

1:  candidates ← V.Search(E(q), top_k=N)            // fast bi-encoder retrieval
2:  scored ← []
3:  for each chunk c ∈ candidates do
4:      relevance ← CE.Score(q, c.content)            // joint cross-encoder scoring
5:      scored.append((c, relevance))
6:  end for
7:  scored ← SortDescending(scored, key=relevance)
8:  return scored[:k]
```

#### 6.1.3 SOTA: ColBERT Late-Interaction Re-Ranking

ColBERT uses a late-interaction mechanism that balances efficiency and cross-attention quality:

$$\text{Score}_{\text{ColBERT}}(q, c) = \sum_{i=1}^{|q|} \max_{j=1}^{|c|} \; \mathbf{q}_i^\top \mathbf{c}_j$$

where $\mathbf{q}_i$ and $\mathbf{c}_j$ are contextualized token embeddings. This MaxSim operation retains token-level interaction without full cross-attention cost.

**Complexity:** $O(|q| \cdot |c|)$ — intermediate between bi-encoder $O(d)$ and cross-encoder $O(|q| \cdot |c| \cdot L)$.

### 6.2 Context Post-Processing

#### 6.2.1 Sentence Window Retrieval — Precision-Context Decoupling

Decouple retrieval granularity from generation context:

**Indexing phase:**

$$\text{For sentence } s_i \text{ in document } D: \quad \text{embed}(s_i), \quad \text{store\_metadata}(W_i)$$

where the window $W_i$ is the surrounding context:

$$W_i = [s_{i-w}, \ldots, s_{i-1}, s_i, s_{i+1}, \ldots, s_{i+w}]$$

with window radius $w$.

**Retrieval phase:** Retrieve by $s_i$ embedding (fine-grained precision), then substitute $s_i \rightarrow W_i$ for generation (broad context).

$$\text{GenContext}(s_i) = W_i \quad \text{where} \quad |W_i|_{\text{tokens}} = (2w + 1) \cdot \overline{|s|}_{\text{tokens}}$$

```
Algorithm 14: SentenceWindowRetrieval
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INDEXING:
Input:  document D, window_radius w, embedding model E
1:  sentences ← SentenceTokenize(D)
2:  for i = 0 to |sentences|-1 do
3:      embedding ← E(sentences[i])
4:      window ← sentences[max(0,i-w) : min(|sentences|, i+w+1)]
5:      Store(embedding, content=sentences[i],
             metadata={window: Join(window), doc_id: D.id, sent_idx: i})
6:  end for

RETRIEVAL:
Input:  query q, top_k k
7:  hits ← VectorSearch(E(q), top_k=k)
8:  generation_contexts ← [hit.metadata.window for hit ∈ hits]
9:  generation_contexts ← Deduplicate(generation_contexts)  // remove overlapping windows
10: return generation_contexts
```

#### 6.2.2 Context Compression — Token-Efficient Evidence Extraction

**Formal objective:** Given retrieved context $\mathcal{D}_{\text{ret}} = \{c_1, \ldots, c_k\}$ with total token count $T_{\text{total}}$, produce compressed context $\mathcal{D}_{\text{comp}}$ such that:

$$|\mathcal{D}_{\text{comp}}|_{\text{tokens}} \leq \gamma \cdot T_{\text{total}}, \quad \gamma \in [0.2, 0.5]$$

while maximizing:

$$\text{InfoRetention}(\mathcal{D}_{\text{comp}}, q) = \frac{\text{RelevantInfo}(\mathcal{D}_{\text{comp}}, q)}{\text{RelevantInfo}(\mathcal{D}_{\text{ret}}, q)} \geq 1 - \epsilon$$

**SOTA Approaches:**

**Approach A — Extractive Compression (Embedding-Based):**

Score each sentence $s_j \in c_i$ for query-relevance:

$$\text{rel}(s_j, q) = \cos\!\big(\mathcal{E}(s_j), \mathcal{E}(q)\big)$$

Retain sentences where $\text{rel}(s_j, q) > \tau_{\text{comp}}$, preserving ordering.

**Approach B — Abstractive Compression (LLM-Based):**

$$\mathcal{D}_{\text{comp}} = \text{LLM}_{\text{compress}}\!\big(\text{"Extract only query-relevant information"}, \; \mathcal{D}_{\text{ret}}, \; q\big)$$

**Approach C — LongLLMLingua (SOTA Token-Level Compression):**

Compute per-token perplexity under a reference language model and remove tokens with lowest information density:

$$\text{InfoDensity}(t_j) = -\log P_{\text{LM}}(t_j \mid t_{<j})$$

Retain tokens with highest information density up to the target budget:

$$\mathcal{D}_{\text{comp}} = \text{TopTokens}\!\big(\mathcal{D}_{\text{ret}}, \; \text{by InfoDensity}, \; \text{budget} = \gamma \cdot T_{\text{total}}\big)$$

```
Algorithm 15: ContextCompression
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:  retrieved chunks D_ret, query q, compression_ratio γ,
        method ∈ {EXTRACTIVE, ABSTRACTIVE, PERPLEXITY}
Output: compressed context D_comp

1:  if method = EXTRACTIVE then
2:      sentences ← FlattenToSentences(D_ret)
3:      scored ← [(s, cos(E(s), E(q))) for s ∈ sentences]
4:      scored ← SortDescending(scored, key=score)
5:      D_comp ← AccumulateUnderBudget(scored, γ · |D_ret|_tokens)
6:      D_comp ← RestoreOriginalOrder(D_comp)
7:  
8:  else if method = PERPLEXITY then
9:      tokens ← Tokenize(D_ret)
10:     perplexities ← ReferenceLM.PerTokenPerplexity(tokens)
11:     // Higher perplexity → more informative → retain
12:     query_relevance ← QueryConditionedScoring(tokens, q)
13:     combined ← perplexities * query_relevance
14:     retain_mask ← TopKMask(combined, budget=γ · |tokens|)
15:     D_comp ← Detokenize(tokens[retain_mask])
16:     
17: else if method = ABSTRACTIVE then
18:     D_comp ← LLM.Generate(
19:         instruction="Extract ONLY query-relevant information. "
20:                     "Preserve factual precision. Remove redundancy.",
21:         context=D_ret, query=q,
22:         max_tokens=floor(γ · |D_ret|_tokens)
23:     )
24: end if
25: return D_comp
```

---

## 7. Prompt Engineering as Compiled Runtime Artifact

### 7.1 Prefill Compilation Architecture

The prompt is not authored — it is compiled from structured components under a deterministic token budget:

$$\text{Prompt}_{\text{compiled}} = \text{Compile}\!\big(\mathcal{P}_{\text{role}}, \; \mathcal{P}_{\text{task}}, \; \mathcal{D}_{\text{comp}}, \; \mathcal{M}_{\text{mem}}, \; \mathcal{T}_{\text{tools}}, \; \mathcal{S}_{\text{state}}\big)$$

subject to:

$$|\text{Prompt}_{\text{compiled}}|_{\text{tokens}} + T_{\text{generation}} \leq T_{\text{context\_window}}$$

### 7.2 Token Budget Allocation

Define the total context window $T_{\text{ctx}}$ and allocate:

$$T_{\text{ctx}} = T_{\text{role}} + T_{\text{task}} + T_{\text{evidence}} + T_{\text{memory}} + T_{\text{tools}} + T_{\text{state}} + T_{\text{gen}}$$

| Component | Typical Budget Share | Priority |
|---|---|---|
| $T_{\text{role}}$ | 3–5% | System instructions, role policy |
| $T_{\text{task}}$ | 5–8% | Task objective, output format spec |
| $T_{\text{evidence}}$ | 40–55% | Retrieved + compressed context |
| $T_{\text{memory}}$ | 10–15% | Session/episodic memory summaries |
| $T_{\text{tools}}$ | 5–10% | Tool schemas, affordance descriptions |
| $T_{\text{state}}$ | 3–5% | Current execution state |
| $T_{\text{gen}}$ | 15–25% | Reserved for generation output |

### 7.3 SOTA Reasoning Frameworks

#### 7.3.1 Chain-of-Thought (CoT)

Instruct the model to produce intermediate reasoning steps:

$$P(r \mid q, \mathcal{D}) = \prod_{t=1}^{T} P(r_t \mid r_{<t}, q, \mathcal{D})$$

CoT inserts explicit reasoning tokens between evidence consumption and answer generation, improving accuracy on multi-step reasoning tasks by approximately 10–25% on benchmarks requiring compositional inference.

#### 7.3.2 Tree-of-Thoughts (ToT)

Extend CoT to explore multiple reasoning branches and evaluate via self-critique:

$$r^* = \arg\max_{r \in \text{Tree}} \; V(r), \quad V(r) = \text{LLM}_{\text{eval}}(\text{"Rate solution quality"}, r)$$

where the tree is explored via BFS or DFS with pruning at each depth level.

**Formal tree construction:**

At each reasoning step $t$, generate $b$ candidate thoughts:

$$\{z_t^{(1)}, \ldots, z_t^{(b)}\} = \text{LLM}.\text{Sample}(s_t, b)$$

Evaluate each candidate:

$$v_t^{(i)} = V(s_t, z_t^{(i)})$$

Prune branches below threshold $\tau_v$ and expand the top-$b'$ branches to the next depth.

#### 7.3.3 ReAct — Reasoning + Acting Agent Loop

$$\text{Loop}: \; \underbrace{\text{Thought}}_{\text{reasoning}} \rightarrow \underbrace{\text{Action}}_{\text{tool call}} \rightarrow \underbrace{\text{Observation}}_{\text{tool result}} \rightarrow \text{Thought} \rightarrow \cdots \rightarrow \text{Answer}$$

Formally:

$$a_t = \pi_\theta(q, \mathcal{D}, h_{<t}), \quad o_t = \text{Tool}(a_t), \quad h_t = h_{<t} \cup \{(a_t, o_t)\}$$

The agent continues until a termination condition is met:

$$\text{Terminate when} \quad \text{LLM}(\text{"Is sufficient evidence gathered?"}, h_t) = \text{True} \quad \text{or} \quad t > t_{\max}$$

```
Algorithm 16: ReActLoop
━━━━━━━━━━━━━━━━━━━━━━
Input:  query q, tools T, max_steps t_max, retrieved context D
Output: final answer

1:  history ← []
2:  for step = 1 to t_max do
3:      thought ← LLM.Generate(
4:          prompt=CompileReActPrompt(q, D, history, T),
5:          stop_at=["Action:", "Answer:"]
6:      )
7:      if thought.contains("Answer:") then
8:          return ExtractAnswer(thought)
9:      end if
10:     action ← ParseAction(thought)
11:     VALIDATE: action.tool ∈ T.registered_tools
12:     observation ← T.Execute(action.tool, action.params, timeout=30s)
13:     history.append({thought, action, observation})
14: end for
15: // Fallback: synthesize from available evidence
16: return LLM.ForcedAnswer(q, D, history)
```

---

## 8. Generative Model Adaptation: Domain-Specific Fine-Tuning

### 8.1 Fine-Tuning Objective

Given a pre-trained LLM $\mathcal{G}_{\theta_0}$ and domain-specific supervised dataset $\mathcal{D}_{\text{ft}} = \{(x_i, y_i)\}_{i=1}^{N}$:

$$\theta^* = \arg\min_\theta \; \frac{1}{N} \sum_{i=1}^{N} \mathcal{L}_{\text{CE}}\!\big(y_i, \; \mathcal{G}_\theta(x_i)\big) + \lambda \|\theta - \theta_0\|_2^2$$

where $\mathcal{L}_{\text{CE}}$ is the cross-entropy loss and the regularization term prevents catastrophic forgetting.

### 8.2 SOTA Fine-Tuning: LoRA and QLoRA

**Low-Rank Adaptation (LoRA)** freezes the pre-trained weights $W_0$ and injects trainable low-rank decomposition:

$$W = W_0 + \Delta W, \quad \Delta W = BA, \quad B \in \mathbb{R}^{d \times r}, \; A \in \mathbb{R}^{r \times d}, \; r \ll d$$

The trainable parameter count reduces from $O(d^2)$ to $O(2dr)$.

**Forward pass:**

$$h = W_0 x + \frac{\alpha}{r} \cdot BAx$$

where $\alpha$ is the scaling factor.

**QLoRA** extends LoRA by quantizing $W_0$ to 4-bit NormalFloat (NF4):

$$W_0^{\text{NF4}} = \text{Quantize}_{\text{NF4}}(W_0)$$

This enables fine-tuning of 65B+ parameter models on single consumer GPUs.

### 8.3 Training Data Construction for RAG Fine-Tuning

| Data Type | Construction Method | Purpose |
|---|---|---|
| $(q, \mathcal{D}_{\text{ret}}, r_{\text{gold}})$ triplets | Human-annotated or GPT-4 distilled | Teach grounded generation from evidence |
| Negative examples with hallucination labels | Synthetic generation + human verification | Train refusal/hedging on insufficient evidence |
| Format-specific examples | Domain templates | Enforce output structure compliance |

### 8.4 Fine-Tuning Pipeline

```
Algorithm 17: RAGSpecificLLMFineTuning
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:  base model G_θ₀, domain dataset D_ft, LoRA config (r, α, target_modules)
Output: fine-tuned model G_θ*

1:  // Prepare training data in RAG format
2:  train_data ← []
3:  for each (qᵢ, dᵢ_gold, rᵢ_gold) ∈ D_ft do
4:      // Simulate retrieval with noise
5:      D_ret ← SimulateRetrieval(qᵢ, C, inject_noise=True)
6:      prompt ← CompileRAGPrompt(qᵢ, D_ret)
7:      train_data.append((prompt, rᵢ_gold))
8:  end for
9:  
10: // Configure LoRA
11: model ← LoadQuantized(G_θ₀, bits=4)           // QLoRA
12: lora_config ← LoRAConfig(r=r, alpha=α,
                              target_modules=["q_proj","v_proj","k_proj","o_proj"])
13: model ← ApplyLoRA(model, lora_config)
14:
15: // Training loop
16: for epoch = 1 to num_epochs do
17:     for batch ∈ Batches(train_data) do
18:         loss ← CrossEntropyLoss(model(batch.prompts), batch.targets)
19:         loss.backward()                         // gradients only on LoRA params
20:         Optimizer.step()
21:     end for
22:     // Evaluation
23:     metrics ← EvaluateRAG(model, val_set)       // faithfulness, relevance, fluency
24:     LOG(epoch, metrics)
25: end for
26: return MergeLoRAWeights(model)                  // merge for deployment
```

### 8.5 Evaluation Metrics for Fine-Tuned RAG Generation

**Faithfulness (Grounding Score):**

$$\text{Faithfulness}(r, \mathcal{D}_{\text{ret}}) = \frac{|\text{Claims}(r) \cap \text{Supported}(\mathcal{D}_{\text{ret}})|}{|\text{Claims}(r)|}$$

**Answer Relevance:**

$$\text{Relevance}(r, q) = \cos\!\big(\mathcal{E}(r), \mathcal{E}(q)\big)$$

**Hallucination Rate:**

$$\text{HalRate}(r, \mathcal{D}_{\text{ret}}) = \frac{|\text{Claims}(r) \setminus \text{Supported}(\mathcal{D}_{\text{ret}})|}{|\text{Claims}(r)|}$$

---

## 9. Unified Pipeline Integration & Evaluation Infrastructure

### 9.1 End-to-End Pipeline Composition

The complete ARAPO pipeline composes as a typed function chain:

$$r = \underbrace{\mathcal{G}_{\theta^*}}_{\text{Gen}} \circ \underbrace{\mathcal{P}_{\text{compile}}}_{\text{Prompt}} \circ \underbrace{\mathcal{K}_{\text{compress}}}_{\text{Compress}} \circ \underbrace{\text{ReRank}_{\text{CE}}}_{\text{ReRank}} \circ \underbrace{\mathcal{R}_{\text{hybrid}}}_{\text{Retrieve}} \circ \underbrace{\text{Route}}_{\text{Route}} \circ \underbrace{\mathcal{Q}_{\text{transform}}}_{\text{PreRetrieval}} \circ \underbrace{\mathcal{I}_{\text{pipeline}}}_{\text{Index}}$$

### 9.2 End-to-End Orchestration Algorithm

```
Algorithm 18: ARAPO-FullPipeline
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:  user query q_raw, corpus C_indexed, session memory M
Output: grounded response r with provenance

PHASE 1: PRE-RETRIEVAL
1:  q_rewritten ← QueryRewrite(q_raw, M)
2:  (π*, q_adapted) ← AgenticQueryRouter(q_rewritten, Π)
3:  if ComplexityScore(q_rewritten) > θ_decompose then
4:      sub_queries ← QueryDecompose(q_adapted)
5:  else
6:      sub_queries ← {q_adapted}
7:  end if

PHASE 2: RETRIEVAL
8:  D_candidates ← ∅
9:  PARALLEL for each sq ∈ sub_queries do
10:     filters ← AutoMetadataFilterExtraction(sq, MetadataSchema)
11:     results ← π*.HybridSearch(sq, C_indexed, filters, over_retrieve=γ·k)
12:     D_candidates ← D_candidates ∪ results
13: end PARALLEL
14: D_candidates ← DeduplicateByContentHash(D_candidates)

PHASE 3: POST-RETRIEVAL
15: D_reranked ← ReRank(D_candidates, q_rewritten, top_k=k)
16: D_autocut ← Autocut(D_reranked, sensitivity=β)
17: D_enhanced ← SentenceWindowExpand(D_autocut)       // if sentence-window indexing
18: D_compressed ← ContextCompress(D_enhanced, q_rewritten, ratio=γ_comp)

PHASE 4: GENERATION
19: prompt ← PrefillCompiler.Compile(
         role_policy=SystemPolicy,
         task=q_raw,
         evidence=D_compressed,
         memory=M.Summarize(budget=T_mem),
         tools=ActiveToolSchemas,
         state=CurrentExecutionState
     )
20: ASSERT |prompt|_tokens + T_gen ≤ T_ctx
21: r ← G_θ*(prompt, temperature=0.1, top_p=0.95)

PHASE 5: VERIFICATION
22: faithfulness ← VerifyGrounding(r, D_compressed)
23: if faithfulness < τ_faith then
24:     r ← G_θ*(AppendCritique(prompt, "Response not grounded. "
                                        "Cite evidence directly."))
25: end if
26:
27: // Provenance attachment
28: r.provenance ← {sources: D_compressed.source_uris,
                     retrieval_scores: D_reranked.scores,
                     pipeline: π*.name, trace_id: GenerateTraceID()}
29:
30: // Memory update (validated writes only)
31: if ContainsNovelCorrection(r, M) then
32:     M.Write(EpisodicMemory, content=r.corrections,
               provenance=r.provenance, expiry=90d)
33: end if
34:
35: return r
```

### 9.3 Continuous Evaluation Infrastructure

#### 9.3.1 RAG Evaluation Framework (RAGAS-Inspired Metrics)

Define the four-dimensional evaluation matrix:

| Metric | Formula | Target |
|---|---|---|
| **Context Precision** | $\frac{|\text{Relevant chunks in top-}k|}{k}$ | $\geq 0.80$ |
| **Context Recall** | $\frac{|\text{Gold facts covered by retrieved context}|}{|\text{All gold facts}|}$ | $\geq 0.90$ |
| **Faithfulness** | $\frac{|\text{Generated claims supported by context}|}{|\text{All generated claims}|}$ | $\geq 0.95$ |
| **Answer Relevance** | $\cos(\mathcal{E}(r), \mathcal{E}(q))$ | $\geq 0.85$ |

#### 9.3.2 Composite Quality Score

$$Q_{\text{RAG}} = w_1 \cdot \text{CtxPrecision} + w_2 \cdot \text{CtxRecall} + w_3 \cdot \text{Faithfulness} + w_4 \cdot \text{AnsRelevance}$$

with default weights $w = [0.2, 0.2, 0.4, 0.2]$ (faithfulness-dominant).

#### 9.3.3 CI/CD Evaluation Pipeline

```
Algorithm 19: ContinuousRAGEvaluation
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Trigger: On every pipeline component change (embedding model, chunking strategy,
         prompt template, LLM version, retrieval config)

1:  benchmark ← LoadBenchmarkSuite()         // curated (q, d_gold, r_gold) triples
2:  for each test_case ∈ benchmark do
3:      r_pred ← ARAPO_FullPipeline(test_case.q)
4:      metrics ← ComputeRAGASMetrics(r_pred, test_case.d_gold, test_case.r_gold)
5:      results.append(metrics)
6:  end for
7:  Q_composite ← WeightedMean(results)
8:  
9:  // Quality gates
10: ASSERT Q_composite.Faithfulness ≥ 0.95,   "FAIL: Faithfulness regression"
11: ASSERT Q_composite.CtxRecall ≥ 0.90,      "FAIL: Recall regression"
12: ASSERT Q_composite.CtxPrecision ≥ 0.80,   "FAIL: Precision regression"
13: ASSERT Latency_p99 ≤ 3000ms,              "FAIL: Latency SLA violation"
14: ASSERT CostPerQuery ≤ budget_threshold,    "FAIL: Cost regression"
15:
16: if ALL_PASS then
17:     DEPLOY(pipeline_version)
18: else
19:     BLOCK_DEPLOY; ALERT(oncall_engineer, failure_report)
20: end if
```

### 9.4 Production Reliability Controls

| Control | Implementation | Purpose |
|---|---|---|
| **Rate Limiting** | Token bucket per-user, per-pipeline | Prevent abuse and cost overrun |
| **Backpressure** | Queue depth monitoring with admission control | Prevent cascade failures under load |
| **Retry with Jitter** | $\text{delay} = \min(b^n + \text{Uniform}(0, j), \text{max\_delay})$ | Prevent thundering herd on transient failures |
| **Circuit Breaker** | Half-open after $t_{\text{cooldown}}$; trip after $n_{\text{fail}}$ consecutive failures | Isolate failing downstream services |
| **Idempotent Mutations** | Idempotency keys on all state-changing tool calls | Safe retry without side-effect duplication |
| **Cache Hierarchy** | L1: in-process embedding cache; L2: distributed retrieval result cache | Reduce redundant computation |
| **Observability** | Structured traces (OpenTelemetry) at every pipeline stage | End-to-end latency attribution and debugging |
| **Graceful Degradation** | Fallback to cached results or reduced-quality pipeline on timeout | Maintain availability under partial failure |

**Retry delay formula with exponential backoff and jitter:**

$$t_{\text{retry}}(n) = \min\!\Big(b^n + U(0, j_{\max}), \; t_{\max}\Big)$$

where $b$ is the backoff base (typically 2), $n$ is the attempt number, $U(0, j_{\max})$ is uniform random jitter, and $t_{\max}$ is the maximum delay cap.

---

## 10. Conclusion & Architectural Invariants

### 10.1 Summary of ARAPO Pipeline Stages and SOTA Techniques

| Pipeline Stage | SOTA Techniques Covered | Key Equation |
|---|---|---|
| **Indexing** | Multimodal extraction (ColPali/ColQwen), MinHash deduplication, schema standardization | $\text{Extract}(d, \tau)$: type-dispatched operator |
| **Chunking** | Semantic chunking (breakpoint detection), Late chunking (context-preserving), Agentic proposition extraction | $\delta_i > \mu_\delta + \alpha \cdot \sigma_\delta$ |
| **Pre-Retrieval** | Trainable query rewriting, diversity-enforced expansion, DAG-based decomposition, agentic routing | $\mathcal{Q}^*(q) = \arg\max \sum \text{RetrievalUtility}$ |
| **Retrieval** | Hybrid search (BM25 + dense), RRF fusion, Autocut outlier exclusion, metadata auto-extraction, embedding fine-tuning (InfoNCE + hard negatives) | $\text{RRF}(c) = \sum_r \frac{1}{\kappa + \text{rank}_r(c)}$ |
| **Post-Retrieval** | Cross-encoder re-ranking, ColBERT late interaction, sentence window retrieval, perplexity-based compression | $\text{Score}_{\text{ColBERT}} = \sum_i \max_j \mathbf{q}_i^\top \mathbf{c}_j$ |
| **Prompt Engineering** | Prefill compilation, CoT, ToT (branching search), ReAct (agent loop) | Token budget: $\sum T_{\text{component}} \leq T_{\text{ctx}}$ |
| **Generation** | QLoRA fine-tuning, faithfulness verification loop, grounding enforcement | $W = W_0 + \frac{\alpha}{r} BA$ |
| **Evaluation** | RAGAS metrics, CI/CD quality gates, continuous benchmark regression | $Q_{\text{RAG}} = \sum w_i \cdot \text{Metric}_i$ |

### 10.2 Architectural Invariants

Every production ARAPO deployment must enforce the following invariants:

1. **Provenance Completeness:** Every generated claim traces to a retrievable, version-stamped source document. No anonymous context blobs.
2. **Token Budget Determinism:** The prefill compiler statically guarantees $|\text{prompt}| + T_{\text{gen}} \leq T_{\text{ctx}}$ before LLM invocation.
3. **Faithfulness Floor:** No pipeline configuration deploys unless $\text{Faithfulness} \geq 0.95$ on the benchmark suite.
4. **Bounded Retrieval Latency:** Hybrid search + re-ranking completes within $T_{\text{retrieval}} \leq 500\text{ms}$ at p99.
5. **Idempotent State Mutations:** All tool calls and memory writes are idempotent-keyed. Retries produce identical outcomes.
6. **Graceful Degradation:** On any component failure, the pipeline degrades to a less capable but still grounded mode rather than producing ungrounded generation.
7. **Continuous Evaluation Coupling:** Every pipeline change triggers automated evaluation. Capability growth is mechanically coupled to quality protection.

---

*End of ARAPO Technical Report.*